# OptimizationSystem 统一 Colab 实验 Notebook

本文件按 TODO 执行顺序合并原有 Notebook。各章节保留原始代码和输出；不同章节可能依赖不同的运行时、模型或量化环境，建议按章节选择性执行。

4080 本地服务器优先使用 `scripts/run_quality_repair_4080.py`，不需要运行本文件。


## 第 1 章：finqa_assets_colab.ipynb

原始文件：`notebooks/finqa_assets_colab.ipynb`


# FinQA 数据资产准备

本 Notebook 为银行金融大模型量化项目准备三类资产：

- `calibration`：用于 AWQ/GPTQ/SmoothQuant 校准；
- `quality_dev` 和 `quality_test`：用于质量评估；
- `high_risk_regression`：固定的金融数值回归集。

数据会缓存到 Google Drive，避免 Colab 运行时重置后重复下载。

In [ ]:
%pip -q install -U datasets huggingface_hub pandas pyarrow transformers==5.18.0 accelerate safetensors tokenizers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 载入项目源码

如果你已经把项目目录上传到 Colab，请修改 `PROJECT_DIR`。如果项目放在 Git 仓库，可以填写 `REPO_URL` 自动克隆。

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = '/content/drive/MyDrive/OptimizationSystem'
REPO_URL = ''

if REPO_URL and not Path(PROJECT_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} 不存在，请上传项目或填写 REPO_URL。'
    )

sys.path.insert(0, PROJECT_DIR)

In [ ]:
import json
import torch
import importlib
import src.input_validation.model_input as model_input_validation
import src.data.finqa_assets as finqa_assets


In [ ]:
import sys
import importlib

for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

importlib.invalidate_caches()
sys.path.insert(0, PROJECT_DIR)

In [ ]:
print('校准集:', len(assets.calibration))
print('质量调优集:', len(assets.quality_dev))
print('最终质量评估集:', len(assets.quality_test))
print('高风险回归集:', len(assets.high_risk_regression))
print('第一条校准 Prompt:')
print(assets.calibration_prompts[0][:2000])

In [ ]:
from src.data.finqa_assets import load_assets

reloaded_assets = load_assets(OUTPUT_DIR)
reloaded_assets.summary()

## TODO 4.1：真实基础模型验收

本节默认使用 `Qwen/Qwen2.5-0.5B-Instruct` 完成真实模型验收。模型约 1 GB，可在 T4 GPU 上以 FP16 运行。代码会固定 Hugging Face commit、下载模型到 Google Drive、计算权重哈希并生成静态与运行时报告。

运行配置单元格前，请阅读模型卡和仓库中的许可证。只有你确认许可证允许当前内部使用范围后，才能把 `INTERNAL_USE_APPROVED` 改成 `True`。

In [ ]:
REAL_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
REAL_MODEL_REVISION = 'main'
INTERNAL_USE_APPROVED = True  # 阅读并确认许可证后手动改为 True
INTERNAL_USAGE_SCOPE = 'Internal research, evaluation, and model optimization only'
REAL_TEST_PROMPT = '请用一句话说明什么是贷款利率。'

print('模型卡: https://huggingface.co/' + REAL_MODEL_ID)
print('内部使用许可已确认:', INTERNAL_USE_APPROVED)

In [ ]:
from huggingface_hub import HfApi, snapshot_download

REAL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_real')
REAL_MODEL_DIR = REAL_ROOT / 'qwen2.5-0.5b-instruct'
REAL_REPORT_DIR = Path('/content/drive/MyDrive/OptimizationSystem/out/model_input_real')
REAL_MANIFEST_PATH = REAL_ROOT / 'model_manifest.json'
REAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
REAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)

model_info = HfApi().model_info(REAL_MODEL_ID, revision=REAL_MODEL_REVISION)
REAL_MODEL_COMMIT = model_info.sha
reported_license = getattr(model_info.card_data, 'license', None) or 'not-declared'
print('解析后的 commit:', REAL_MODEL_COMMIT)
print('模型卡声明的许可证:', reported_license)

snapshot_download(
    repo_id=REAL_MODEL_ID,
    revision=REAL_MODEL_COMMIT,
    local_dir=REAL_MODEL_DIR,
    allow_patterns=[
        '*.json', '*.safetensors', '*.model', '*.tiktoken',
        'vocab.*', 'merges.*', 'LICENSE*',
    ],
)
print('真实模型目录:', REAL_MODEL_DIR)

In [ ]:
weight_files = sorted(
    str(path.relative_to(REAL_MODEL_DIR))
    for pattern in ('*.safetensors', 'pytorch_model*.bin')
    for path in REAL_MODEL_DIR.rglob(pattern)
    if path.is_file()
)
tokenizer_names = (
    'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model',
    'vocab.json', 'merges.txt', 'special_tokens_map.json',
)
tokenizer_files = [
    name for name in tokenizer_names if (REAL_MODEL_DIR / name).is_file()
]
license_files = sorted(
    path.name for path in REAL_MODEL_DIR.glob('LICENSE*') if path.is_file()
)
if not license_files:
    raise FileNotFoundError('模型仓库没有下载到 LICENSE 文件，不能完成许可证检查。')

real_manifest = {
    'schema_version': 1,
    'model_id': REAL_MODEL_ID,
    'model_path': str(REAL_MODEL_DIR),
    'version': 'Qwen2.5-0.5B-Instruct',
    'commit': REAL_MODEL_COMMIT,
    'weight_files': weight_files,
    'config_file': 'config.json',
    'tokenizer_files': tokenizer_files,
    'expected_sha256': {},
    'license': {
        'name': reported_license,
        'file': license_files[0],
        'internal_use_approved': INTERNAL_USE_APPROVED,
        'usage_scope': INTERNAL_USAGE_SCOPE,
    },
    'runtime': {
        'framework': 'transformers',
        'model_class': 'causal_lm',
        'dtype': 'float16',
        'device': 'cuda:0',
        'trust_remote_code': False,
        'smoke_test_prompt': REAL_TEST_PROMPT,
    },
}
REAL_MANIFEST_PATH.write_text(
    json.dumps(real_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Manifest:', REAL_MANIFEST_PATH)
print('权重文件:', weight_files)
print('Tokenizer 文件:', tokenizer_files)
print('许可证文件:', license_files[0])

In [ ]:
real_static_report = model_input_validation.validate_model_input(
    REAL_MANIFEST_PATH
)
real_static_payload = real_static_report.to_dict()
(REAL_REPORT_DIR / 'static.json').write_text(
    json.dumps(real_static_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(real_static_payload, ensure_ascii=False, indent=2))
assert real_static_report.validation_passed, (
    '静态检查未通过。若仅许可证项失败，请确认许可证后把 '
    'INTERNAL_USE_APPROVED 改为 True，并重新运行配置、manifest 和本单元格。'
)

In [ ]:
import gc

if not torch.cuda.is_available():
    raise RuntimeError('请先把 Colab 运行时切换为 T4 GPU。')
gc.collect()
torch.cuda.empty_cache()

real_runtime_report = model_input_validation.validate_model_input(
    REAL_MANIFEST_PATH, runtime_check=True
)
real_runtime_payload = real_runtime_report.to_dict()
(REAL_REPORT_DIR / 'runtime.json').write_text(
    json.dumps(real_runtime_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(real_runtime_payload, ensure_ascii=False, indent=2))
assert real_runtime_report.complete, '真实模型存在未通过项，请查看上面的 detail。'
print('TODO 4.1 真实模型验收：PASS')
print('报告目录:', REAL_REPORT_DIR)

## TODO 4.2：评测资产输入验收

本节重新生成增强版数据 manifest，实际加载校准集、开发/最终评估集和高风险回归集，检查路径、版本、隔离、字段、指标接口、关键错误规则、样本数与 SHA-256。数据仍保存在原 `banking_llm_project/datasets` 路径，只有验收结果写入 `OptimizationSystem/out/dataset_input`。

In [ ]:
import importlib

importlib.reload(finqa_assets)
prepare_assets = finqa_assets.prepare_assets
assets = prepare_assets(
    output_dir=OUTPUT_DIR,
    calibration_size=1024,
    seed=42,
)
DATASET_MANIFEST_PATH = Path(OUTPUT_DIR) / 'manifest.json'
print('增强版数据 Manifest:', DATASET_MANIFEST_PATH)
print(json.dumps(assets.manifest, ensure_ascii=False, indent=2))

In [ ]:
import shutil
import src.input_validation.dataset_input as dataset_input_validation

importlib.reload(dataset_input_validation)
dataset_report = dataset_input_validation.validate_dataset_input(
    DATASET_MANIFEST_PATH
)
dataset_payload = dataset_report.to_dict()
DATASET_RESULT_DIR = Path(
    '/content/drive/MyDrive/OptimizationSystem/out/dataset_input'
)
DATASET_RESULT_DIR.mkdir(parents=True, exist_ok=True)
(DATASET_RESULT_DIR / 'validation.json').write_text(
    json.dumps(dataset_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
shutil.copy2(
    DATASET_MANIFEST_PATH,
    DATASET_RESULT_DIR / 'dataset_manifest.json',
)
print(json.dumps(dataset_payload, ensure_ascii=False, indent=2))
assert dataset_report.complete, '评测资产存在未通过项，请查看上面的 detail。'
print('TODO 4.2 评测资产输入验收：PASS')
print('结果目录:', DATASET_RESULT_DIR)

## TODO 4.3：硬件环境输入验收

本节采集 Colab 的真实 NVIDIA GPU、显存、驱动、CUDA、容器、PyTorch、Transformers 和拓扑信息。硬件 manifest 保存在原资产目录 `banking_llm_project/hardware`，项目结果保存在 `OptimizationSystem/out/hardware`。

> Colab 没有昇腾设备，因此本节只能完成 NVIDIA 部分。Todo 4.3 必须等真实昇腾机器的 manifest 合并通过后才能整体勾选。

In [ ]:
import importlib
import src.input_validation.hardware_input as hardware_input_validation

importlib.reload(hardware_input_validation)
HARDWARE_ASSET_DIR = Path(
    '/content/drive/MyDrive/banking_llm_project/hardware'
)
HARDWARE_RESULT_DIR = Path(
    '/content/drive/MyDrive/OptimizationSystem/out/hardware'
)
NVIDIA_MANIFEST_PATH = HARDWARE_ASSET_DIR / 'nvidia_colab.json'
HARDWARE_ASSET_DIR.mkdir(parents=True, exist_ok=True)
HARDWARE_RESULT_DIR.mkdir(parents=True, exist_ok=True)

nvidia_manifest = hardware_input_validation.collect_nvidia_environment()
NVIDIA_MANIFEST_PATH.write_text(
    json.dumps(nvidia_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(nvidia_manifest, ensure_ascii=False, indent=2))
assert nvidia_manifest['accelerator']['available'], '没有检测到 NVIDIA GPU。'
print('NVIDIA 环境采集：PASS')
print('硬件 Manifest:', NVIDIA_MANIFEST_PATH)

In [ ]:
import shutil

nvidia_only_report = hardware_input_validation.validate_hardware_inputs(
    NVIDIA_MANIFEST_PATH
)
nvidia_only_payload = nvidia_only_report.to_dict()
(HARDWARE_RESULT_DIR / 'nvidia_validation.json').write_text(
    json.dumps(nvidia_only_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
shutil.copy2(
    NVIDIA_MANIFEST_PATH,
    HARDWARE_RESULT_DIR / 'nvidia_colab.json',
)
print(json.dumps(nvidia_only_payload, ensure_ascii=False, indent=2))
assert nvidia_only_payload['checks'][0]['status'] == 'pass'
assert nvidia_only_payload['checks'][1]['status'] == 'pass'
assert nvidia_only_report.complete is False
print('NVIDIA 两项验收：PASS；昇腾与双平台合并验收：PENDING')
print('结果目录:', HARDWARE_RESULT_DIR)

## TODO 5.2：FP16 基线推理服务

本节在真实 T4 上加载 Qwen2.5-0.5B-Instruct，执行单请求、两条样本批量推理和确定性重放，并把请求级性能日志、模型版本及运行环境写入 `OptimizationSystem/out/baseline`。模型和 FinQA 数据仍从原有路径读取，不写入该结果目录。

In [ ]:
import importlib
import shutil

import src.baseline.service as baseline_service

importlib.reload(baseline_service)
BASELINE_CONFIG_PATH = Path(PROJECT_DIR) / 'configs/baseline.json'
MODEL_MANIFEST_PATH = Path(
    '/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json'
)
if not MODEL_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f'找不到真实模型 manifest: {MODEL_MANIFEST_PATH}。'
        '请确认 Google Drive 已挂载，并先完成 TODO 4.1。'
    )
BASELINE_RESULT_DIR = Path(
    '/content/drive/MyDrive/OptimizationSystem/out/baseline'
)
BASELINE_LOG_PATH = BASELINE_RESULT_DIR / 'request_logs.jsonl'
BASELINE_RESULT_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_LOG_PATH.unlink(missing_ok=True)

baseline = baseline_service.BaselineService.from_local_model(
    config_file=BASELINE_CONFIG_PATH,
    model_manifest_file=MODEL_MANIFEST_PATH,
    log_file=BASELINE_LOG_PATH,
)
print('FP16 基线模型加载：PASS')
print(json.dumps(baseline.model_metadata, ensure_ascii=False, indent=2))
print(json.dumps(baseline.environment_metadata(), ensure_ascii=False, indent=2))

In [ ]:
if 'assets' not in globals():
    assets = finqa_assets.load_assets(OUTPUT_DIR)

single_request = baseline_service.InferenceRequest.from_finqa_row(
    assets.quality_dev[0],
    request_id='baseline-single-0001',
)
batch_requests = [
    baseline_service.InferenceRequest.from_finqa_row(
        assets.quality_dev[index],
        request_id=f'baseline-batch-{index + 1:04d}',
    )
    for index in range(2)
]
single_result = baseline.generate_one(single_request)
batch_results = baseline.generate_batch(batch_requests)
replay_result = baseline.generate_one(single_request)
replay_match = (
    single_result['output_text'] == replay_result['output_text']
)
assert len(batch_results) == 2
assert replay_match, '固定种子重放输出不一致。'
print('单请求输出:', single_result['output_text'])
print('批量输出:', [item['output_text'] for item in batch_results])
print('确定性重放一致:', replay_match)

In [ ]:
baseline_config = json.loads(
    BASELINE_CONFIG_PATH.read_text(encoding='utf-8')
)
log_records = [
    json.loads(line)
    for line in BASELINE_LOG_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
required_log_fields = {
    'request_id', 'success', 'batch_size', 'input_tokens',
    'output_tokens', 'latency_ms', 'output_tokens_per_second',
    'model_id', 'model_commit', 'dtype', 'device',
}
baseline_evidence = {
    'model_loaded': True,
    'config': baseline_config,
    'single_request': {
        'success': single_result['success'] is True,
        'request_id': single_result['request_id'],
        'output_text': single_result['output_text'],
    },
    'batch_request': {
        'success': len(batch_results) == 2,
        'result_count': len(batch_results),
        'request_ids': [item['request_id'] for item in batch_results],
    },
    'deterministic_replay_match': replay_match,
    'request_logs': {
        'path': str(BASELINE_LOG_PATH),
        'record_count': len(log_records),
        'all_success': all(row.get('success') is True for row in log_records),
        'required_fields_present': all(
            required_log_fields.issubset(row) for row in log_records
        ),
    },
    'model_metadata': baseline.model_metadata,
    'environment': baseline.environment_metadata(),
}
evidence_path = BASELINE_RESULT_DIR / 'baseline_evidence.json'
evidence_path.write_text(
    json.dumps(baseline_evidence, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
shutil.copy2(
    BASELINE_CONFIG_PATH,
    BASELINE_RESULT_DIR / 'baseline_config.json',
)
baseline_report = baseline_service.validate_baseline_evidence(evidence_path)
baseline_payload = baseline_report.to_dict()
(BASELINE_RESULT_DIR / 'validation.json').write_text(
    json.dumps(baseline_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(baseline_payload, ensure_ascii=False, indent=2))
assert baseline_report.complete, '基线服务存在未通过项，请查看 detail。'
print('TODO 5.2 基线服务验收：PASS')
print('结果目录:', BASELINE_RESULT_DIR)

## TODO 5.3：统一 FP16 Benchmark

本节在同一 Tesla T4、同一 FP16 模型和统一生成方式下，测试 3 种 batch size、2 种输入长度和 2 种输出长度，共 12 个测试点。每个测试点预热 1 次、正式测量 5 次，并记录 TTFT、单 token 延迟、端到端 P50/P95/P99、tokens/s、requests/s、GPU 峰值显存和实际 KV Cache 张量显存。

In [ ]:
import importlib
import shutil

import src.baseline.service as baseline_service
import src.benchmark.runner as benchmark_runner
import src.data.finqa_assets as finqa_assets

importlib.reload(benchmark_runner)
BENCHMARK_CONFIG_PATH = Path(PROJECT_DIR) / 'configs/benchmark.json'
MODEL_MANIFEST_PATH = (
    Path(PROJECT_DIR) / 'out/model_input_real/model_manifest.json'
)
BENCHMARK_RESULT_DIR = Path(PROJECT_DIR) / 'out/benchmark'
BENCHMARK_RESULT_DIR.mkdir(parents=True, exist_ok=True)
benchmark_config = json.loads(
    BENCHMARK_CONFIG_PATH.read_text(encoding='utf-8')
)
if 'baseline' not in globals():
    baseline = baseline_service.BaselineService.from_local_model(
        config_file=Path(PROJECT_DIR) / 'configs/baseline.json',
        model_manifest_file=MODEL_MANIFEST_PATH,
        log_file=BENCHMARK_RESULT_DIR / 'service.jsonl',
    )
if 'assets' not in globals():
    assets = finqa_assets.load_assets(OUTPUT_DIR)
benchmark_request = baseline_service.InferenceRequest.from_finqa_row(
    assets.quality_dev[0], request_id='benchmark-workload'
)
benchmark_prompt = benchmark_request.render_prompt()
(BENCHMARK_RESULT_DIR / 'benchmark_prompt.txt').write_text(
    benchmark_prompt, encoding='utf-8'
)
benchmark_cases = benchmark_runner.build_cases(benchmark_config)
print('测试点数量:', len(benchmark_cases))
print('总生成调用次数:', len(benchmark_cases) * (
    benchmark_config['warmup_runs'] + benchmark_config['measured_runs']
))
print('结果目录:', BENCHMARK_RESULT_DIR)

In [ ]:
if 'benchmark_report' not in globals():
    benchmark_report = benchmark_runner.run_benchmark(
        baseline, benchmark_prompt, benchmark_config
    )
else:
    print('复用当前 Colab 会话中已完成的 Benchmark 结果。')
benchmark_report_path = (
    BENCHMARK_RESULT_DIR / 'baseline_fp16_report.json'
)
benchmark_report_path.write_text(
    json.dumps(benchmark_report, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
shutil.copy2(
    BENCHMARK_CONFIG_PATH,
    BENCHMARK_RESULT_DIR / 'benchmark_config.json',
)
benchmark_validation = benchmark_runner.validate_benchmark_report(
    benchmark_report_path
)
benchmark_validation_payload = benchmark_validation.to_dict()
(BENCHMARK_RESULT_DIR / 'validation.json').write_text(
    json.dumps(benchmark_validation_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
benchmark_summary = [
    {
        'case_id': case['case_id'],
        'ttft_p50_ms': case['latency_ms']['ttft']['p50'],
        'e2e_p50_ms': case['latency_ms']['end_to_end']['p50'],
        'e2e_p95_ms': case['latency_ms']['end_to_end']['p95'],
        'e2e_p99_ms': case['latency_ms']['end_to_end']['p99'],
        'tokens_per_second': case['throughput']['mean_output_tokens_per_second'],
        'requests_per_second': case['throughput']['mean_requests_per_second'],
        'peak_allocated_mib': case['memory']['peak_allocated_mib'],
        'kv_cache_mib': case['memory']['kv_cache_mib'],
    }
    for case in benchmark_report['cases']
]
print(json.dumps(benchmark_summary, ensure_ascii=False, indent=2))
print(json.dumps(benchmark_validation_payload, ensure_ascii=False, indent=2))
assert benchmark_validation.complete, (
    'Benchmark 存在未通过项，请查看 validation.json。'
)
print('TODO 5.3 Benchmark 验收：PASS')
print('结果目录:', BENCHMARK_RESULT_DIR)

## TODO 5.4：环境锁定

本节锁定真实 Tesla T4 基线环境，包括 Python、PyTorch、Transformers、CUDA、cuDNN、驱动、Colab 容器版本及完整 `pip freeze`。昇腾/CANN 必须等真实昇腾机器采集，基础容器必须实际构建并保存镜像 digest，因此这三项继续标记为 `not_run`。

In [ ]:
import importlib
import sys
from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/OptimizationSystem"
MODULE_FILE = Path(PROJECT_DIR) / "src/environment_lock/ma.py"

print("模块文件存在:", MODULE_FILE.is_file())
assert MODULE_FILE.is_file(), MODULE_FILE

# 确保项目目录优先
sys.path = [path for path in sys.path if path != PROJECT_DIR]
sys.path.insert(0, PROJECT_DIR)

# 清除已经加载的旧 src 包
for module_name in list(sys.modules):
  if module_name == "src" or module_name.startswith("src."):
    del sys.modules[module_name]

importlib.invalidate_caches()

In [ ]:
import importlib
import shutil

import src.environment_lock.manager as environment_lock_manager

importlib.reload(environment_lock_manager)
ENVIRONMENT_RESULT_DIR = Path(PROJECT_DIR) / 'out/environment_lock'
ENVIRONMENT_RESULT_DIR.mkdir(parents=True, exist_ok=True)
NVIDIA_HARDWARE_PATH = (
    Path(PROJECT_DIR) / 'out/hardware/nvidia_colab.json'
)
BENCHMARK_REPORT_PATH = (
    Path(PROJECT_DIR) / 'out/benchmark/baseline_fp16_report.json'
)
DEPENDENCY_LOCK_PATH = (
    ENVIRONMENT_RESULT_DIR / 'requirements.freeze.txt'
)
NVIDIA_LOCK_PATH = (
    ENVIRONMENT_RESULT_DIR / 'nvidia_environment.json'
)
ASCEND_PENDING_PATH = (
    ENVIRONMENT_RESULT_DIR / 'ascend_environment.pending.json'
)
environment_lock_manager.capture_pip_freeze(DEPENDENCY_LOCK_PATH)
nvidia_lock = environment_lock_manager.create_nvidia_lock(
    hardware_manifest_file=NVIDIA_HARDWARE_PATH,
    benchmark_report_file=BENCHMARK_REPORT_PATH,
    dependency_lock_file=DEPENDENCY_LOCK_PATH,
    output_file=NVIDIA_LOCK_PATH,
)
shutil.copy2(
    Path(PROJECT_DIR) / 'configs/environment/ascend.pending.json',
    ASCEND_PENDING_PATH,
)
print('NVIDIA 环境锁定文件:', NVIDIA_LOCK_PATH)
print('依赖包数量:', nvidia_lock['dependencies']['package_count'])
print('pip freeze SHA-256:', nvidia_lock['dependencies']['sha256'])
print(json.dumps({
    'runtime': nvidia_lock['runtime'],
    'software': nvidia_lock['software'],
    'frameworks': nvidia_lock['frameworks'],
}, ensure_ascii=False, indent=2))

In [ ]:
environment_report = environment_lock_manager.validate_environment_lock(
    nvidia_lock_file=NVIDIA_LOCK_PATH,
    dependency_lock_file=DEPENDENCY_LOCK_PATH,
    ascend_lock_file=ASCEND_PENDING_PATH,
)
environment_payload = environment_report.to_dict()
(ENVIRONMENT_RESULT_DIR / 'validation.json').write_text(
    json.dumps(environment_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
statuses = {
    check['name']: check['status']
    for check in environment_payload['checks']
}
expected_pass = {
    'nvidia_environment_file_created',
    'python_torch_transformers_versions_pinned',
    'inference_framework_version_pinned',
    'dependency_lock_saved',
    'hardware_and_driver_recorded',
}
expected_not_run = {
    'ascend_environment_file_created',
    'cuda_and_cann_versions_pinned',
    'base_container_image_built',
}
assert all(statuses[name] == 'pass' for name in expected_pass)
assert all(statuses[name] == 'not_run' for name in expected_not_run)
assert environment_report.complete is False
print(json.dumps(environment_payload, ensure_ascii=False, indent=2))
print('TODO 5.4 NVIDIA 环境锁定：PASS')
print('昇腾/CANN/基础容器构建：PENDING')
print('结果目录:', ENVIRONMENT_RESULT_DIR)

### 5.5 质量实验环境对齐

质量实验必须与已完成的 Benchmark 使用同一套 Transformers 版本。当前锁定版本为 `5.18.0`。如果下面单元格提示版本不一致，请先安装指定版本并重启 Colab 运行时，再重新运行前置导入、基线服务和质量实验单元。

In [ ]:
import importlib.metadata
EXPECTED_TRANSFORMERS_VERSION = '5.18.0'
actual_transformers_version = importlib.metadata.version('transformers')
print('当前 Transformers:', actual_transformers_version)
if actual_transformers_version != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f'版本不一致：需要 {EXPECTED_TRANSFORMERS_VERSION}，'
        f'当前为 {actual_transformers_version}。请先运行：\n'
        f'%pip install -q --upgrade transformers=={EXPECTED_TRANSFORMERS_VERSION}\n'
        '然后重启 Colab 运行时，并重新运行项目导入、5.2 和 5.3 的前置单元。'
    )
print('Benchmark/质量实验 Transformers 版本对齐：PASS')

## TODO 5.5：FP16 质量基线与 NVIDIA 阶段验收

本节在版本化 FinQA development 集的全部 883 条样本上运行 FP16 模型，保存逐样本预测并计算数值准确率和解析率。该结果作为后续量化与剪枝方案的质量参照；最终 test 集继续保留，不在本阶段使用。

In [ ]:
import json
from pathlib import Path
import importlib
import shutil

import src.baseline.service as baseline_service
import src.data.finqa_assets as finqa_assets
import src.evaluation.baseline_quality as baseline_quality
import src.evaluation.stage5_acceptance as stage5_acceptance

importlib.reload(baseline_quality)
importlib.reload(stage5_acceptance)
QUALITY_CONFIG_PATH = (
    Path(PROJECT_DIR) / 'configs/baseline_quality.json'
)
QUALITY_RESULT_DIR = Path(PROJECT_DIR) / 'out/baseline_quality_experiment'
STAGE5_RESULT_DIR = Path(PROJECT_DIR) / 'out/stage5'
QUALITY_RESULT_DIR.mkdir(parents=True, exist_ok=True)
STAGE5_RESULT_DIR.mkdir(parents=True, exist_ok=True)
quality_config = json.loads(
    QUALITY_CONFIG_PATH.read_text(encoding='utf-8')
)
quality_log_path = QUALITY_RESULT_DIR / 'quality_service.jsonl'
quality_log_path.unlink(missing_ok=True)
if 'baseline' in globals():
    quality_service = baseline_service.BaselineService(
        model=baseline.model,
        tokenizer=baseline.tokenizer,
        torch_module=baseline.torch,
        config=baseline.config,
        model_metadata=baseline.model_metadata,
        log_file=quality_log_path,
    )
else:
    quality_service = baseline_service.BaselineService.from_local_model(
        config_file=Path(PROJECT_DIR) / 'configs/baseline.json',
        model_manifest_file=(
            Path(PROJECT_DIR) / 'out/model_input_real/model_manifest.json'
        ),
        log_file=quality_log_path,
    )
if 'assets' not in globals():
    assets = finqa_assets.load_assets(OUTPUT_DIR)
print('评测样本数:', len(assets.quality_dev))
print('batch size:', quality_config['batch_size'])
print('质量结果目录:', QUALITY_RESULT_DIR)

In [ ]:
import json
from pathlib import Path
quality_report = baseline_quality.run_baseline_quality(
    service=quality_service,
    dataset=assets.quality_dev,
    dataset_manifest=assets.manifest,
    quality_config=quality_config,
    output_dir=QUALITY_RESULT_DIR,
)
quality_report_path = (
    QUALITY_RESULT_DIR / 'baseline_quality_report.json'
)
quality_report_path.write_text(
    json.dumps(quality_report, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
shutil.copy2(
    QUALITY_CONFIG_PATH, QUALITY_RESULT_DIR / 'baseline_quality_config.json'
)
quality_validation = baseline_quality.validate_quality_report(
    quality_report_path
)
quality_validation_payload = quality_validation.to_dict()
(QUALITY_RESULT_DIR / 'validation.json').write_text(
    json.dumps(quality_validation_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(quality_report['metrics'], ensure_ascii=False, indent=2))
print(json.dumps(quality_validation_payload, ensure_ascii=False, indent=2))
assert quality_validation.complete, (
    'FP16 质量基线存在未通过项，请拉回报告后分析。'
)
print('FP16 全量开发集质量基线：PASS')

In [ ]:
import json
from pathlib import Path
stage5_report = stage5_acceptance.validate_stage5(
    model_report_file=(
        Path(PROJECT_DIR) / 'out/model_input_real/reports/runtime.json'
    ),
    baseline_validation_file=(
        Path(PROJECT_DIR) / 'out/baseline/validation.json'
    ),
    benchmark_validation_file=(
        Path(PROJECT_DIR) / 'out/benchmark/validation.json'
    ),
    benchmark_report_file=(
        Path(PROJECT_DIR) / 'out/benchmark/baseline_fp16_report.json'
    ),
    quality_validation_file=(
        QUALITY_RESULT_DIR / 'validation.json'
    ),
    quality_report_file=quality_report_path,
    environment_validation_file=(
        Path(PROJECT_DIR) / 'out/environment_lock/validation.json'
    ),
    dataset_validation_file=(
        Path(PROJECT_DIR) / 'out/dataset_input/validation.json'
    ),
    business_validation_file=(
        Path(PROJECT_DIR) / 'out/business_scope/validation.json'
    ),
    deliverable_files={
        'baseline_model_run_instructions': (
            Path(PROJECT_DIR) / 'docs/BASELINE_SERVICE.md'
        ),
        'baseline_quality_report': quality_report_path,
        'baseline_performance_report': (
            Path(PROJECT_DIR) / 'out/benchmark/baseline_fp16_report.json'
        ),
        'input_interface_document': (
            Path(PROJECT_DIR) / 'docs/INPUT_INTERFACE.md'
        ),
        'environment_lock_file': (
            Path(PROJECT_DIR) / 'out/environment_lock/nvidia_environment.json'
        ),
    },
)
stage5_payload = stage5_report.to_dict()
(STAGE5_RESULT_DIR / 'validation.json').write_text(
    json.dumps(stage5_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(stage5_payload, ensure_ascii=False, indent=2))
assert stage5_report.complete, '第 5 章 NVIDIA 阶段验收未通过。'
print('TODO 5.5 NVIDIA 基线阶段验收：PASS')
print('质量结果目录:', QUALITY_RESULT_DIR)
print('阶段验收目录:', STAGE5_RESULT_DIR)

## TODO 6.1：统一量化框架接入

本节统一 AWQ、GPTQ 和 SmoothQuant 的配置、校准资产引用、导出文件名和 JSONL 日志字段，并采集当前 Colab 后端包版本。真实量化权重导出不在本节伪造，留到 6.2～6.4。

In [40]:
import importlib
import json
from pathlib import Path

import src.quantization.framework as quantization_framework

importlib.reload(quantization_framework)
QUANTIZATION_CONFIG_PATH = Path(PROJECT_DIR) / 'configs/quantization.json'
CALIBRATION_MANIFEST_PATH = (
    Path(PROJECT_DIR) / 'out/dataset_input/dataset_manifest.json'
)
QUANTIZATION_RESULT_DIR = Path(PROJECT_DIR) / 'out/quantization_framework'
QUANTIZATION_RESULT_DIR.mkdir(parents=True, exist_ok=True)
framework_evidence = quantization_framework.collect_framework_evidence(
    config_file=QUANTIZATION_CONFIG_PATH,
    calibration_manifest_file=CALIBRATION_MANIFEST_PATH,
    output_dir=QUANTIZATION_RESULT_DIR,
)
framework_evidence_path = QUANTIZATION_RESULT_DIR / 'framework_evidence.json'
framework_evidence_path.write_text(
    json.dumps(framework_evidence, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps({
    'backend_packages': framework_evidence['backend_packages'],
    'backend_available': framework_evidence['backend_available'],
    'status': framework_evidence['status'],
}, ensure_ascii=False, indent=2))

{
  "backend_packages": {
    "autoawq": null,
    "auto-gptq": null,
    "smoothquant": null,
    "torch": "2.11.0+cu130",
    "transformers": "5.18.0"
  },
  "backend_available": {
    "awq": false,
    "gptq": false,
    "smoothquant": false
  },
  "status": "framework_ready_not_quantized"
}


In [41]:
quantization_report = quantization_framework.validate_quantization_evidence(
    framework_evidence_path
)
quantization_payload = quantization_report.to_dict()
(QUANTIZATION_RESULT_DIR / 'validation.json').write_text(
    json.dumps(quantization_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(quantization_payload, ensure_ascii=False, indent=2))
assert all(
    check['status'] in {'pass', 'not_run'}
    for check in quantization_payload['checks']
)
print('TODO 6.1 配置/导出/日志契约验收：PASS')
print('真实量化执行状态:', framework_evidence['status'])
print('结果目录:', QUANTIZATION_RESULT_DIR)

{
  "evidence": "/content/drive/MyDrive/OptimizationSystem/out/quantization_framework/framework_evidence.json",
  "complete": false,
  "checks": [
    {
      "name": "awq_gptq_smoothquant_configs",
      "status": "pass",
      "detail": "AWQ, GPTQ, and SmoothQuant share one validated configuration contract."
    },
    {
      "name": "unified_export_contract",
      "status": "pass",
      "detail": "Quantized artifacts use one manifest and config naming contract."
    },
    {
      "name": "unified_quantization_logging",
      "status": "pass",
      "detail": "All adapters declare the same JSONL event-log fields."
    },
    {
      "name": "quantization_tool_versions_recorded",
      "status": "pass",
      "detail": "Core PyTorch and Transformers versions are recorded."
    },
    {
      "name": "real_quantization_execution",
      "status": "not_run",
      "detail": "Actual backend execution and exported weights are deferred to 6.2-6.4."
    }
  ]
}
TODO 6.1 配置/导出/日志契约验收：PAS

## 第 2 章：awq_experiment_colab.ipynb

原始文件：`notebooks/awq_experiment_colab.ipynb`


# TODO 6.2：AWQ INT4 实验

本 Notebook 使用真实 Qwen 模型和 `calibration_finqa_1024`，测试 AWQ W4A16 的 group size 32、64、128。完整量化模型写入 `banking_llm_project/awq`，只有配置、manifest 和事件日志写入 `OptimizationSystem/out/awq`。

请先完成 4.1、4.2，并使用 T4 GPU。AutoAWQ 安装后通常需要重启 Colab 运行时，再从头运行本 Notebook。

In [ ]:
%pip -q install autoawq
# 安装后若提示需要重启运行时，请重启，再运行后续单元。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib
import json
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CONFIG_PATH = PROJECT_DIR / 'configs/awq.json'
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
RESULT_DIR = PROJECT_DIR / 'out/awq'
MODEL_RESULT_DIR = Path('/content/drive/MyDrive/banking_llm_project/awq')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_DIR))
import src.data.finqa_assets as finqa_assets
import src.quantization.awq as awq
importlib.reload(awq)
assets = finqa_assets.load_assets(DATASET_DIR)
print('校准样本:', len(assets.calibration))
print('结果目录:', RESULT_DIR)

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plans = awq.build_awq_plans(config)
print([plan.to_dict() for plan in plans])
assert MODEL_MANIFEST.is_file(), f'找不到模型 manifest: {MODEL_MANIFEST}'
assert CALIBRATION_MANIFEST.is_file(), f'找不到校准 manifest: {CALIBRATION_MANIFEST}'
print('AWQ 配置和输入资产：PASS')

In [ ]:
# 如果之前把完整模型导出到了 out/awq，可运行本单元迁移非分析文件。
import shutil
for plan in plans:
    old_dir = RESULT_DIR / plan.plan_id
    new_dir = MODEL_RESULT_DIR / plan.plan_id
    new_dir.mkdir(parents=True, exist_ok=True)
    for pattern in ('*.safetensors', '*.bin', 'config.json', 'generation_config.json', 'tokenizer*', 'special_tokens_map.json'):
        for source in old_dir.glob(pattern):
            shutil.move(str(source), str(new_dir / source.name))
    manifest_file = old_dir / 'quantized_model_manifest.json'
    if manifest_file.is_file():
        manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
        manifest['model_path'] = str(new_dir.resolve())
        manifest['analysis_artifacts'] = {
            'config': str((old_dir / 'quantization_config.json').resolve()),
            'manifest': str(manifest_file.resolve()),
            'events': str((old_dir / 'quantization_events.jsonl').resolve()),
        }
        manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        shutil.copy2(old_dir / 'quantization_config.json', new_dir / 'quantization_config.json')
        shutil.copy2(manifest_file, new_dir / 'quantized_model_manifest.json')
print('已将模型文件迁移到:', MODEL_RESULT_DIR)

## 真实执行
下面单元格会依次执行三个 group size。每个计划都可能占用数分钟和较多显存；不要把输出目录改到模型目录。

In [ ]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError('请切换到 NVIDIA T4 GPU。')
execution_results = []
for plan in plans:
    plan_dir = RESULT_DIR / plan.plan_id
    model_plan_dir = MODEL_RESULT_DIR / plan.plan_id
    print(f'开始 AWQ: {plan.plan_id}')
    execution_results.append(awq.run_awq_plan(
        plan=plan,
        model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=plan_dir,
        model_output_dir=model_plan_dir,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
    ))
    del plan_dir
    gc.collect()
    torch.cuda.empty_cache()
print(json.dumps(execution_results, ensure_ascii=False, indent=2))

In [ ]:
evidence = awq.collect_awq_evidence(
    config_file=CONFIG_PATH,
    calibration_manifest_file=CALIBRATION_MANIFEST,
    model_manifest_file=MODEL_MANIFEST,
    output_dir=RESULT_DIR,
)
evidence_path = RESULT_DIR / 'awq_evidence.json'
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = awq.validate_awq_evidence(evidence_path)
validation_path = RESULT_DIR / 'validation.json'
validation_path.write_text(json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=2))
assert report.complete, 'AWQ 三个 group size 未全部完成，请查看 validation.json。'
print('TODO 6.2 AWQ 导出验收：PASS')
print('结果目录:', RESULT_DIR)

## 第 3 章：awq_quality_benchmark_colab.ipynb

原始文件：`notebooks/awq_quality_benchmark_colab.ipynb`


# TODO 6.2：AWQ 质量、回归和性能评估

本 Notebook 使用已经导出的 AWQ g32/g64/g128 模型，复用项目现有 FinQA 指标和 Benchmark。完整模型从 `banking_llm_project/awq` 读取，评测结果写入 `OptimizationSystem/out/awq`。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/baseline_quality.json'
BENCHMARK_CONFIG = PROJECT_DIR / 'configs/benchmark.json'
AWQ_RESULT_DIR = PROJECT_DIR / 'out/awq'
BASELINE_PREDICTIONS = PROJECT_DIR / 'out/baseline_quality_experiment/baseline_predictions.jsonl'
sys.path.insert(0, str(PROJECT_DIR))
from src.data.finqa_assets import load_assets
from src.evaluation.baseline_quality import run_baseline_quality, validate_quality_report
from src.evaluation.finqa_metrics import classify_numeric_regression
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.quantization.awq import load_awq_service
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
benchmark_config = json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
print('质量集:', len(assets.quality_dev), '回归集:', len(assets.high_risk_regression))

Mounted at /content/drive
质量集: 883 回归集: 91


In [2]:
%pip -q install autoawq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
baseline_rows = [json.loads(line) for line in BASELINE_PREDICTIONS.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(baseline_rows) == len(assets.quality_dev), 'FP16 基线预测数量不完整。'
baseline_by_question = {row['question']: row for row in baseline_rows}
print('FP16 基线预测：PASS')

FP16 基线预测：PASS


In [4]:
from src.baseline.service import BaselineService, InferenceRequest
baseline_service = BaselineService.from_local_model(
    config_file=BASELINE_CONFIG,
    model_manifest_file=PROJECT_DIR / 'out/model_input_real/model_manifest.json',
    log_file=AWQ_RESULT_DIR / 'fp16_regression_service.jsonl',
)
def predict_rows(service, dataset, prefix):
    rows = []
    for start in range(0, len(dataset), int(quality_config['batch_size'])):
        stop = min(start + int(quality_config['batch_size']), len(dataset))
        batch = [dataset[index] for index in range(start, stop)]
        requests = [InferenceRequest.from_finqa_row(row, request_id=f'{prefix}-{start + offset + 1:06d}') for offset, row in enumerate(batch)]
        for row, request, result in zip(batch, requests, service.generate_batch(requests, prompt_suffix=quality_config.get('prompt_suffix', ''))):
            rows.append({'question': request.question, 'prediction': result['output_text'], 'reference': row.get('answer') or row.get('qa', {}).get('answer')})
    return rows
baseline_regression_rows = predict_rows(baseline_service, assets.high_risk_regression, 'fp16-regression')
print('FP16 高风险回归基线：', len(baseline_regression_rows))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

FP16 高风险回归基线： 91


In [5]:
for plan_id in ('awq-int4-w4a16-g32', 'awq-int4-w4a16-g64', 'awq-int4-w4a16-g128'):
    plan_dir = AWQ_RESULT_DIR / plan_id
    manifest_file = plan_dir / 'quantized_model_manifest.json'
    quality_dir = plan_dir / 'quality'
    service = load_awq_service(
        baseline_config_file=BASELINE_CONFIG,
        quantized_manifest_file=manifest_file,
        source_model_manifest_file=MODEL_MANIFEST,
        log_file=plan_dir / 'awq_quality_service.jsonl',
    )
    quality_report = run_baseline_quality(
        service=service, dataset=assets.quality_dev,
        dataset_manifest=assets.manifest, quality_config=quality_config,
        output_dir=quality_dir,
    )
    quality_report_path = quality_dir / 'baseline_quality_report.json'
    quality_report_path.write_text(json.dumps(quality_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    quality_validation = validate_quality_report(quality_report_path)
    (quality_dir / 'validation.json').write_text(json.dumps(quality_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    predictions = [json.loads(line) for line in (quality_dir / 'baseline_predictions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    regressions = []
    for row in predictions:
        baseline = baseline_by_question.get(row['question'])
        if baseline is None:
            continue
        regressions.append({**classify_numeric_regression(baseline['prediction'], row['prediction'], row['reference'], float(quality_config.get('tolerance', 1e-4))), 'question': row['question']})
    regression_payload = {
        'schema_version': 1, 'plan_id': plan_id, 'dataset': quality_report['dataset'],
        'total': len(regressions), 'new_regressions': sum(row['new_regression'] for row in regressions),
        'critical_errors': sum(row['critical_error_type'] is not None for row in regressions),
        'rows': regressions,
    }
    (plan_dir / 'regression.json').write_text(json.dumps(regression_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(plan_id, quality_report['metrics'], 'new_regressions=', regression_payload['new_regressions'])

/usr/local/lib/python3.13/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

Quality evaluation: 4/883


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality evaluation: 108/883
Quality evaluation: 112/883
Quality evaluation: 116/883
Quality evaluation: 120/883
Quality evaluation: 124/883
Quality evaluation: 128/883
Quality evaluation: 132/883
Quality evaluation: 136/883
Quality evaluation: 140/883
Quality evaluation: 144/883
Quality evaluation: 148/883
Quality evaluati

Replacing layers...: 100%|██████████| 24/24 [00:06<00:00,  3.62it/s]


Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality evaluation: 108/883
Quality evaluation: 112/883
Quality evaluation: 116/883
Quality evaluation: 120/883
Quality evaluation: 124/883
Quality evaluation: 128/883
Quality evaluation: 132/883
Quality evaluation: 136/883
Quality evaluation: 140/883
Quality evaluation: 144/883
Quality evaluation

Replacing layers...: 100%|██████████| 24/24 [00:06<00:00,  3.99it/s]


Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality evaluation: 108/883
Quality evaluation: 112/883
Quality evaluation: 116/883
Quality evaluation: 120/883
Quality evaluation: 124/883
Quality evaluation: 128/883
Quality evaluation: 132/883
Quality evaluation: 136/883
Quality evaluation: 140/883
Quality evaluation: 144/883
Quality evaluation

In [6]:
for plan_id in ('awq-int4-w4a16-g32', 'awq-int4-w4a16-g64', 'awq-int4-w4a16-g128'):
    plan_dir = AWQ_RESULT_DIR / plan_id
    candidate_service = load_awq_service(BASELINE_CONFIG, plan_dir / 'quantized_model_manifest.json', MODEL_MANIFEST, plan_dir / 'awq_regression_service.jsonl')
    candidate_rows = predict_rows(candidate_service, assets.high_risk_regression, f'{plan_id}-regression')
    baseline_map = {row['question']: row for row in baseline_regression_rows}
    comparisons = []
    for row in candidate_rows:
        baseline_row = baseline_map[row['question']]
        comparisons.append({**classify_numeric_regression(baseline_row['prediction'], row['prediction'], row['reference'], float(quality_config.get('tolerance', 1e-4))), 'question': row['question']})
    regression_payload = {'schema_version': 1, 'plan_id': plan_id, 'dataset_role': 'high_risk_regression', 'total': len(comparisons), 'new_regressions': sum(row['new_regression'] for row in comparisons), 'critical_errors': sum(row['critical_error_type'] is not None for row in comparisons), 'rows': comparisons}
    (plan_dir / 'regression.json').write_text(json.dumps(regression_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(plan_id, 'high-risk regressions=', regression_payload['new_regressions'])

Replacing layers...: 100%|██████████| 24/24 [00:06<00:00,  3.81it/s]


awq-int4-w4a16-g32 high-risk regressions= 0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Replacing layers...: 100%|██████████| 24/24 [00:06<00:00,  3.73it/s]


awq-int4-w4a16-g64 high-risk regressions= 0


Replacing layers...: 100%|██████████| 24/24 [00:06<00:00,  3.73it/s]


awq-int4-w4a16-g128 high-risk regressions= 0


In [7]:
for plan_id in ('awq-int4-w4a16-g32', 'awq-int4-w4a16-g64', 'awq-int4-w4a16-g128'):
    plan_dir = AWQ_RESULT_DIR / plan_id
    manifest_file = plan_dir / 'quantized_model_manifest.json'
    service = load_awq_service(BASELINE_CONFIG, manifest_file, MODEL_MANIFEST, plan_dir / 'awq_benchmark_service.jsonl')
    benchmark_report = run_benchmark(service, assets.calibration_prompts[0], benchmark_config)
    report_file = plan_dir / 'benchmark_report.json'
    report_file.write_text(json.dumps(benchmark_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    benchmark_validation = validate_benchmark_report(report_file)
    (plan_dir / 'benchmark_validation.json').write_text(json.dumps(benchmark_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(plan_id, 'benchmark:', benchmark_validation.complete)

Replacing layers...: 100%|██████████| 24/24 [00:06<00:00,  3.78it/s]


Benchmark 1/12: bs1-in128-out16
Benchmark 2/12: bs1-in128-out32


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Benchmark 3/12: bs1-in512-out16
Benchmark 4/12: bs1-in512-out32
Benchmark 5/12: bs2-in128-out16
Benchmark 6/12: bs2-in128-out32
Benchmark 7/12: bs2-in512-out16
Benchmark 8/12: bs2-in512-out32
Benchmark 9/12: bs4-in128-out16
Benchmark 10/12: bs4-in128-out32
Benchmark 11/12: bs4-in512-out16
Benchmark 12/12: bs4-in512-out32
awq-int4-w4a16-g32 benchmark: True


Replacing layers...: 100%|██████████| 24/24 [00:06<00:00,  3.74it/s]


Benchmark 1/12: bs1-in128-out16
Benchmark 2/12: bs1-in128-out32
Benchmark 3/12: bs1-in512-out16
Benchmark 4/12: bs1-in512-out32
Benchmark 5/12: bs2-in128-out16
Benchmark 6/12: bs2-in128-out32
Benchmark 7/12: bs2-in512-out16
Benchmark 8/12: bs2-in512-out32
Benchmark 9/12: bs4-in128-out16
Benchmark 10/12: bs4-in128-out32
Benchmark 11/12: bs4-in512-out16
Benchmark 12/12: bs4-in512-out32
awq-int4-w4a16-g64 benchmark: True


Replacing layers...: 100%|██████████| 24/24 [00:06<00:00,  3.73it/s]


Benchmark 1/12: bs1-in128-out16
Benchmark 2/12: bs1-in128-out32
Benchmark 3/12: bs1-in512-out16
Benchmark 4/12: bs1-in512-out32
Benchmark 5/12: bs2-in128-out16
Benchmark 6/12: bs2-in128-out32
Benchmark 7/12: bs2-in512-out16
Benchmark 8/12: bs2-in512-out32
Benchmark 9/12: bs4-in128-out16
Benchmark 10/12: bs4-in128-out32
Benchmark 11/12: bs4-in512-out16
Benchmark 12/12: bs4-in512-out32
awq-int4-w4a16-g128 benchmark: True


完成后请将每个 plan 目录中的 `quality/`、`regression.json`、`benchmark_report.json` 和两个 JSONL 服务日志保留在 `out/awq`，模型权重仍只保留在 `banking_llm_project/awq`。

## 第 4 章：gptq_experiment_colab.ipynb

原始文件：`notebooks/gptq_experiment_colab.ipynb`


# TODO 6.3：GPTQ INT4 实验

使用与 AWQ 相同的校准集、bit 数和 group size 32/64/128。完整模型写入 `banking_llm_project/gptq`，分析文件写入 `OptimizationSystem/out/gptq`。

首次运行先执行安装单元，然后通过 `运行时 → 重启会话` 重启。重启后跳过安装单元，从挂载 Drive 单元继续。

In [5]:
%pip -q install -U gptqmodel==7.5.0 optimum accelerate safetensors
%pip -q install -U transformers==5.18.0
print('安装完成。现在必须重启 Colab 会话，不要继续运行下一个单元。')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 133.7 MB/s eta 0:00:00
安装完成。现在必须重启 Colab 会话，不要继续运行下一个单元。


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import importlib.metadata
import json
import sys
from pathlib import Path
import torch

print('transformers:', importlib.metadata.version('transformers'))
print('gptqmodel:', importlib.metadata.version('gptqmodel'))
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig
print('Transformers GPTQ 依赖导入：PASS')

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CONFIG_PATH = PROJECT_DIR / 'configs/gptq.json'
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
RESULT_DIR = PROJECT_DIR / 'out/gptq'
MODEL_RESULT_DIR = Path('/content/drive/MyDrive/banking_llm_project/gptq')
sys.path.insert(0, str(PROJECT_DIR))

from src.data.finqa_assets import load_assets
from src.quantization.gptq import (
    build_gptq_plans, collect_gptq_evidence,
    run_gptq_plan, validate_gptq_evidence,
)

assets = load_assets(DATASET_DIR)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plans = build_gptq_plans(config)
assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
print(torch.cuda.get_device_name(0), [plan.plan_id for plan in plans])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
transformers: 5.18.0
gptqmodel: 7.5.0
Transformers GPTQ 依赖导入：PASS
NVIDIA A100-SXM4-40GB ['gptq-int4-w4a16-g32', 'gptq-int4-w4a16-g64', 'gptq-int4-w4a16-g128']


In [2]:
import gc

results = []
for plan in plans:
    manifest = RESULT_DIR / plan.plan_id / 'quantized_model_manifest.json'
    if manifest.is_file():
        print('跳过已完成:', plan.plan_id)
        continue
    result = run_gptq_plan(
        plan=plan,
        model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=RESULT_DIR / plan.plan_id,
        model_output_dir=MODEL_RESULT_DIR / plan.plan_id,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
    )
    results.append(result)
    if result['status'] != 'pass':
        raise RuntimeError(result['detail'])
    gc.collect()
    torch.cuda.empty_cache()
print(json.dumps(results, ensure_ascii=False, indent=2))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.5.0
Transformers : 5.18.0
Torch        : 2.11.0+cu130
Triton       : 3.6.0


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Quantizing model.layers blocks :   0%|          | 0/24 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

...

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

INFO  Packing Kernel: Auto-selection: adding candidate `TritonV2Linear`         
INFO  Kernel: selected -> `TritonV2Linear`.                                     
INFO  gc.collect() reclaimed 317 objects in 0.333s                              


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[
  {
    "plan": {
      "plan_id": "gptq-int4-w4a16-g32",
      "method": "gptq",
      "bits": 4,
      "weight_bits": 4,
      "activation_bits": 16,
      "group_size": 32,
      "quantization_granularity": "groupwise",
      "backend_package": "gptqmodel"
    },
    "status": "pass",
    "output_dir": "/content/drive/MyDrive/OptimizationSystem/out/gptq/gptq-int4-w4a16-g32",
    "model_output_dir": "/content/drive/MyDrive/banking_llm_project/gptq/gptq-int4-w4a16-g32"
  },
  {
    "plan": {
      "plan_id": "gptq-int4-w4a16-g64",
      "method": "gptq",
      "bits": 4,
      "weight_bits": 4,
      "activation_bits": 16,
      "group_size": 64,
      "quantization_granularity": "groupwise",
      "backend_package": "gptqmodel"
    },
    "status": "pass",
    "output_dir": "/content/drive/MyDrive/OptimizationSystem/out/gptq/gptq-int4-w4a16-g64",
    "model_output_dir": "/content/drive/MyDrive/banking_llm_project/gptq/gptq-int4-w4a16-g64"
  },
  {
    "plan": {
      "plan_id": "gp

In [3]:
evidence = collect_gptq_evidence(
    CONFIG_PATH, CALIBRATION_MANIFEST, MODEL_MANIFEST, RESULT_DIR
)
evidence_path = RESULT_DIR / 'gptq_evidence.json'
evidence_path.write_text(
    json.dumps(evidence, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
report = validate_gptq_evidence(evidence_path)
(RESULT_DIR / 'validation.json').write_text(
    json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=2))
assert report.complete, 'GPTQ 三个 group size 未全部完成。'
print('TODO 6.3 GPTQ 导出验收：PASS')

{
  "evidence": "/content/drive/MyDrive/OptimizationSystem/out/gptq/gptq_evidence.json",
  "complete": true,
  "checks": [
    {
      "name": "gptq_awq_comparable_matrix",
      "status": "pass",
      "detail": "GPTQ uses INT4 group sizes 32, 64, and 128."
    },
    {
      "name": "gptq_backend_recorded",
      "status": "pass",
      "detail": "GPTQModel version is recorded."
    },
    {
      "name": "gptq_quantized_artifacts",
      "status": "pass",
      "detail": "Each GPTQ plan has config, manifest, and event log."
    },
    {
      "name": "gptq_real_execution",
      "status": "pass",
      "detail": "All GPTQ plans completed real export."
    }
  ]
}
TODO 6.3 GPTQ 导出验收：PASS


In [4]:
import gc, json, sys
from pathlib import Path
import torch
PROJECT_DIR=Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR=Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST=Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG=PROJECT_DIR/'configs/baseline.json'
QUALITY_CONFIG=PROJECT_DIR/'configs/baseline_quality.json'
BENCHMARK_CONFIG=PROJECT_DIR/'configs/quantization_benchmark.json'
GPTQ_DIR=PROJECT_DIR/'out/gptq'
AWQ_DIR=PROJECT_DIR/'out/awq'
sys.path.insert(0,str(PROJECT_DIR))
from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.evaluation.baseline_quality import run_baseline_quality, validate_quality_report
from src.evaluation.finqa_metrics import classify_numeric_regression
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.quantization.gptq import load_gptq_service
from src.quantization.awq import load_awq_service
assets=load_assets(DATASET_DIR)
quality_config=json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
benchmark_config=json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
print(torch.cuda.get_device_name(0),len(assets.quality_dev),len(assets.high_risk_regression))

NVIDIA A100-SXM4-40GB 883 91


In [5]:
def predict_rows(service,dataset,prefix):
    rows=[]; batch_size=int(quality_config['batch_size'])
    for start in range(0,len(dataset),batch_size):
        batch=[dataset[index] for index in range(start,min(start+batch_size,len(dataset)))]
        requests=[InferenceRequest.from_finqa_row(row,request_id=f'{prefix}-{start+offset+1:06d}') for offset,row in enumerate(batch)]
        results=service.generate_batch(requests,prompt_suffix=quality_config.get('prompt_suffix',''))
        for row,request,result in zip(batch,requests,results):
            rows.append({'question':request.question,'prediction':result['output_text'],'reference':row.get('answer') or row.get('qa',{}).get('answer')})
    return rows
baseline=BaselineService.from_local_model(BASELINE_CONFIG,PROJECT_DIR/'out/model_input_real/model_manifest.json',GPTQ_DIR/'fp16_regression_service.jsonl')
baseline_regression=predict_rows(baseline,assets.high_risk_regression,'fp16-regression')
del baseline; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
for group_size in (32,64,128):
    plan_id=f'gptq-int4-w4a16-g{group_size}'; plan_dir=GPTQ_DIR/plan_id
    service=load_gptq_service(BASELINE_CONFIG,plan_dir/'quantized_model_manifest.json',MODEL_MANIFEST,plan_dir/'gptq_quality_service.jsonl')
    quality_dir=plan_dir/'quality'
    quality=run_baseline_quality(service,assets.quality_dev,assets.manifest,quality_config,quality_dir)
    quality_file=quality_dir/'baseline_quality_report.json'
    quality_file.write_text(json.dumps(quality,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    validation=validate_quality_report(quality_file)
    (quality_dir/'validation.json').write_text(json.dumps(validation.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    candidate=predict_rows(service,assets.high_risk_regression,f'{plan_id}-regression')
    baseline_map={row['question']:row for row in baseline_regression}
    comparisons=[{**classify_numeric_regression(baseline_map[row['question']]['prediction'],row['prediction'],row['reference'],float(quality_config.get('tolerance',1e-4))),'question':row['question']} for row in candidate]
    regression={'schema_version':1,'plan_id':plan_id,'dataset_role':'high_risk_regression','total':len(comparisons),'new_regressions':sum(row['new_regression'] for row in comparisons),'critical_errors':sum(row['critical_error_type'] is not None for row in comparisons),'rows':comparisons}
    (plan_dir/'regression.json').write_text(json.dumps(regression,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    print(plan_id,quality['metrics'],regression['new_regressions'])
    del service; gc.collect(); torch.cuda.empty_cache()

INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  Kernel: selected -> `MarlinLinear`.                                       


Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  Marlin fp16: compiling torch.ops JIT extension in `/root/.cache/gptqmodel/torch_extensions/marlin_fp16/3380075ecccdabdf`.
INFO  Marlin fp16: torch.ops JIT extension ready in 184s (estimated ~117s, +67s).
INFO  gc.collect() reclaimed 7327 objects in 0.330s                             
Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality

Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6735 objects in 0.321s                             
Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality evaluation: 108/883
Quality evaluation: 112/883
Quality evaluation: 116/883
Quality evaluation: 120/883
Quality evaluation: 124/883
Quality evaluation: 128/883
Quality evaluation: 132/883
Quality evaluation: 1

Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6735 objects in 0.331s                             
Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality evaluation: 108/883
Quality evaluation: 112/883
Quality evaluation: 116/883
Quality evaluation: 120/883
Quality evaluation: 124/883
Quality evaluation: 128/883
Quality evaluation: 132/883
Quality evaluation: 1

In [8]:
%pip -q install autoawq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [9]:
for method,root,loader in [('gptq',GPTQ_DIR,load_gptq_service),('awq',AWQ_DIR,load_awq_service)]:
    for group_size in (32,64,128):
        plan_id=f'{method}-int4-w4a16-g{group_size}'; plan_dir=root/plan_id
        service=loader(BASELINE_CONFIG,plan_dir/'quantized_model_manifest.json',MODEL_MANIFEST,plan_dir/f'{method}_comparison_benchmark_service.jsonl')
        report=run_benchmark(service,assets.calibration_prompts[0],benchmark_config)
        report_file=plan_dir/'comparison_benchmark_report.json'
        report_file.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        validation=validate_benchmark_report(report_file)
        (plan_dir/'comparison_benchmark_validation.json').write_text(json.dumps(validation.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        assert validation.complete
        print(plan_id,'comparison benchmark PASS')
        del service; gc.collect(); torch.cuda.empty_cache()

INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  Kernel: selected -> `MarlinLinear`.                                       


Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6765 objects in 0.328s                             
Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
gptq-int4-w4a16-g32 comparison benchmark PASS
INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  Kernel: selected -> `MarlinLinear`.                                       


Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6735 objects in 0.321s                             
Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
gptq-int4-w4a16-g64 comparison benchmark PASS
INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  Kernel: selected -> `MarlinLinear`.                                       


...

Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
awq-int4-w4a16-g32 comparison benchmark PASS


Replacing layers...: 100%|██████████| 24/24 [00:07<00:00,  3.00it/s]


Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
awq-int4-w4a16-g64 comparison benchmark PASS


Replacing layers...: 100%|██████████| 24/24 [00:08<00:00,  3.00it/s]


Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
awq-int4-w4a16-g128 comparison benchmark PASS


In [10]:
%pip -q install -U smoothquant

In [14]:
import json, sys
from pathlib import Path
PROJECT_DIR=Path('/content/drive/MyDrive/OptimizationSystem')
CONFIG_PATH=PROJECT_DIR/'configs/int8_fp8.json'
CALIBRATION_MANIFEST=PROJECT_DIR/'out/dataset_input/dataset_manifest.json'
RESULT_DIR=PROJECT_DIR/'out/int8_fp8'
sys.path.insert(0,str(PROJECT_DIR))
from src.quantization.int8_fp8 import collect_int8_fp8_evidence, validate_int8_fp8_evidence
evidence=collect_int8_fp8_evidence(CONFIG_PATH,CALIBRATION_MANIFEST,RESULT_DIR)
evidence_path=RESULT_DIR/'int8_fp8_evidence.json'
evidence_path.write_text(json.dumps(evidence,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
report=validate_int8_fp8_evidence(evidence_path)
(RESULT_DIR/'validation.json').write_text(json.dumps(report.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(report.to_dict(),ensure_ascii=False,indent=2))

{
  "evidence": "/content/drive/MyDrive/OptimizationSystem/out/int8_fp8/int8_fp8_evidence.json",
  "complete": false,
  "checks": [
    {
      "name": "smoothquant_int8_config",
      "status": "pass",
      "detail": "SmoothQuant W8A8 configuration and calibration are recorded."
    },
    {
      "name": "smoothquant_export_contract",
      "status": "pass",
      "detail": "INT8 export and event-log names are fixed."
    },
    {
      "name": "nvidia_fp8_capability",
      "status": "pass",
      "detail": "NVIDIA CUDA exposes a supported PyTorch FP8 dtype."
    },
    {
      "name": "ascend_fp8_capability",
      "status": "not_run",
      "detail": "Ascend NPU and FP8 dtype support are detected."
    },
    {
      "name": "smoothquant_real_execution",
      "status": "not_run",
      "detail": "SmoothQuant INT8 weights were exported."
    }
  ]
}


In [13]:
import sys
import importlib

for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

importlib.invalidate_caches()
sys.path.insert(0, PROJECT_DIR)

## 第 5 章：gptq_quality_benchmark_colab.ipynb

原始文件：`notebooks/gptq_quality_benchmark_colab.ipynb`


# TODO 6.3：GPTQ 质量与 AWQ 性能对比

对 GPTQ g32/g64/g128 执行完整 FinQA 质量和高风险回归，并在同一张 GPU 上对 GPTQ/AWQ 执行 batch 1/4/8/16 的统一 Benchmark。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import gc, json, sys
from pathlib import Path
import torch
PROJECT_DIR=Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR=Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST=Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG=PROJECT_DIR/'configs/baseline.json'
QUALITY_CONFIG=PROJECT_DIR/'configs/baseline_quality.json'
BENCHMARK_CONFIG=PROJECT_DIR/'configs/quantization_benchmark.json'
GPTQ_DIR=PROJECT_DIR/'out/gptq'
AWQ_DIR=PROJECT_DIR/'out/awq'
sys.path.insert(0,str(PROJECT_DIR))
from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.evaluation.baseline_quality import run_baseline_quality, validate_quality_report
from src.evaluation.finqa_metrics import classify_numeric_regression
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.quantization.gptq import load_gptq_service
from src.quantization.awq import load_awq_service
assets=load_assets(DATASET_DIR)
quality_config=json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
benchmark_config=json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
print(torch.cuda.get_device_name(0),len(assets.quality_dev),len(assets.high_risk_regression))

In [ ]:
def predict_rows(service,dataset,prefix):
    rows=[]; batch_size=int(quality_config['batch_size'])
    for start in range(0,len(dataset),batch_size):
        batch=[dataset[index] for index in range(start,min(start+batch_size,len(dataset)))]
        requests=[InferenceRequest.from_finqa_row(row,request_id=f'{prefix}-{start+offset+1:06d}') for offset,row in enumerate(batch)]
        results=service.generate_batch(requests,prompt_suffix=quality_config.get('prompt_suffix',''))
        for row,request,result in zip(batch,requests,results):
            rows.append({'question':request.question,'prediction':result['output_text'],'reference':row.get('answer') or row.get('qa',{}).get('answer')})
    return rows
baseline=BaselineService.from_local_model(BASELINE_CONFIG,PROJECT_DIR/'out/model_input_real/model_manifest.json',GPTQ_DIR/'fp16_regression_service.jsonl')
baseline_regression=predict_rows(baseline,assets.high_risk_regression,'fp16-regression')
del baseline; gc.collect(); torch.cuda.empty_cache()

In [ ]:
for group_size in (32,64,128):
    plan_id=f'gptq-int4-w4a16-g{group_size}'; plan_dir=GPTQ_DIR/plan_id
    service=load_gptq_service(BASELINE_CONFIG,plan_dir/'quantized_model_manifest.json',MODEL_MANIFEST,plan_dir/'gptq_quality_service.jsonl')
    quality_dir=plan_dir/'quality'
    quality=run_baseline_quality(service,assets.quality_dev,assets.manifest,quality_config,quality_dir)
    quality_file=quality_dir/'baseline_quality_report.json'
    quality_file.write_text(json.dumps(quality,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    validation=validate_quality_report(quality_file)
    (quality_dir/'validation.json').write_text(json.dumps(validation.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    candidate=predict_rows(service,assets.high_risk_regression,f'{plan_id}-regression')
    baseline_map={row['question']:row for row in baseline_regression}
    comparisons=[{**classify_numeric_regression(baseline_map[row['question']]['prediction'],row['prediction'],row['reference'],float(quality_config.get('tolerance',1e-4))),'question':row['question']} for row in candidate]
    regression={'schema_version':1,'plan_id':plan_id,'dataset_role':'high_risk_regression','total':len(comparisons),'new_regressions':sum(row['new_regression'] for row in comparisons),'critical_errors':sum(row['critical_error_type'] is not None for row in comparisons),'rows':comparisons}
    (plan_dir/'regression.json').write_text(json.dumps(regression,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    print(plan_id,quality['metrics'],regression['new_regressions'])
    del service; gc.collect(); torch.cuda.empty_cache()

In [ ]:
for method,root,loader in [('gptq',GPTQ_DIR,load_gptq_service),('awq',AWQ_DIR,load_awq_service)]:
    for group_size in (32,64,128):
        plan_id=f'{method}-int4-w4a16-g{group_size}'; plan_dir=root/plan_id
        service=loader(BASELINE_CONFIG,plan_dir/'quantized_model_manifest.json',MODEL_MANIFEST,plan_dir/f'{method}_comparison_benchmark_service.jsonl')
        report=run_benchmark(service,assets.calibration_prompts[0],benchmark_config)
        report_file=plan_dir/'comparison_benchmark_report.json'
        report_file.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        validation=validate_benchmark_report(report_file)
        (plan_dir/'comparison_benchmark_validation.json').write_text(json.dumps(validation.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        assert validation.complete
        print(plan_id,'comparison benchmark PASS')
        del service; gc.collect(); torch.cuda.empty_cache()

完成后拉回 `out/gptq`，并拉回 `out/awq/*/comparison_benchmark_*`。完整 GPTQ/AWQ 权重不要放进项目仓库。

## 第 6 章：int8_fp8_experiment_colab.ipynb

原始文件：`notebooks/int8_fp8_experiment_colab.ipynb`


# TODO 6.4：SmoothQuant INT8 / FP8 能力验证

使用 NVIDIA ModelOpt 对真实模型执行 W8A8 SmoothQuant。FP8 只有在目标设备成功完成真实 FP8 矩阵运算后才记为 `pass`；仅存在 PyTorch FP8 dtype 不算通过。完整 INT8 状态写入 `banking_llm_project`，分析证据写入 `OptimizationSystem/out`。

In [1]:
%pip -q install -U "nvidia-modelopt[torch]" accelerate safetensors
%pip -q install -U transformers==5.18.0
print('安装完成。请重启 Colab 会话；重启后跳过本单元。')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 48.0 MB/s eta 0:00:00
安装完成。请重启 Colab 会话；重启后跳过本单元。


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
CONFIG_PATH = PROJECT_DIR / 'configs/int8_fp8.json'
RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
MODEL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/int8_fp8')
sys.path.insert(0, str(PROJECT_DIR))

from src.data.finqa_assets import load_assets
from src.quantization.int8_fp8 import (
    build_smoothquant_plan, collect_int8_fp8_evidence,
    run_smoothquant_plan, validate_int8_fp8_evidence,
)
assert torch.cuda.is_available(), '请切换到 NVIDIA GPU 运行时。'
assets = load_assets(DATASET_DIR)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plan = build_smoothquant_plan(config)
print('GPU:', torch.cuda.get_device_name(0))
print('校准样本:', len(assets.calibration_prompts))
print('计划:', plan.to_dict())

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
校准样本: 1024
计划: {'plan_id': 'smoothquant-int8-w8a8', 'method': 'smoothquant', 'weight_bits': 8, 'activation_bits': 8, 'alpha': 0.5, 'quantization_granularity': 'per_channel_weight_per_tensor_activation', 'backend_package': 'nvidia-modelopt'}


In [3]:
plan_result_dir = RESULT_DIR / plan.plan_id
plan_model_dir = MODEL_ROOT / plan.plan_id
manifest = plan_result_dir / 'quantized_model_manifest.json'
if manifest.is_file():
    print('跳过已完成导出:', manifest)
else:
    result = run_smoothquant_plan(
        plan=plan,
        model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=plan_result_dir,
        model_output_dir=plan_model_dir,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
        calibration_batch_size=int(config['calibration_batch_size']),
    )
    if result['status'] != 'pass':
        raise RuntimeError(result.get('detail', result))
    gc.collect(); torch.cuda.empty_cache()
print('INT8 manifest:', manifest)

/usr/local/lib/python3.13/dist-packages/modelopt/torch/__init__.py:55: UserWarning: transformers 5.18.0 is not tested with current version of modelopt and may cause issues. Please install recommended version with `pip install -U nvidia-modelopt[hf]` if working with HF models.
  _warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Registered <class 'transformers.models.qwen2.modeling_qwen2.Qwen2Attention'> to _QuantAttention for KV Cache quantization
Inserted 606 quantizers
Smoothed 168 modules


/usr/local/lib/python3.13/dist-packages/modelopt/torch/quantization/model_calib.py:1342: UserWarning: lm_head is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")


INT8 manifest: /content/drive/MyDrive/OptimizationSystem/out/int8_fp8/smoothquant-int8-w8a8/quantized_model_manifest.json


In [4]:
evidence = collect_int8_fp8_evidence(CONFIG_PATH, CALIBRATION_MANIFEST, RESULT_DIR)
evidence_path = RESULT_DIR / 'int8_fp8_evidence.json'
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = validate_int8_fp8_evidence(evidence_path)
payload = report.to_dict()
(RESULT_DIR / 'validation.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(payload, ensure_ascii=False, indent=2))
statuses = {item['name']: item['status'] for item in payload['checks']}
assert statuses['smoothquant_real_execution'] == 'pass'
print('SmoothQuant INT8 真实导出：PASS')
print('NVIDIA FP8:', statuses['nvidia_fp8_capability'])
print('Ascend FP8:', statuses['ascend_fp8_capability'])

{
  "evidence": "/content/drive/MyDrive/OptimizationSystem/out/int8_fp8/int8_fp8_evidence.json",
  "complete": false,
  "checks": [
    {
      "name": "smoothquant_int8_config",
      "status": "pass",
      "detail": "SmoothQuant W8A8 configuration and calibration are recorded."
    },
    {
      "name": "smoothquant_export_contract",
      "status": "pass",
      "detail": "INT8 export and event-log names are fixed."
    },
    {
      "name": "smoothquant_backend_recorded",
      "status": "pass",
      "detail": "NVIDIA ModelOpt version is recorded."
    },
    {
      "name": "smoothquant_quantized_artifacts",
      "status": "pass",
      "detail": "SmoothQuant config, manifest, and event log exist."
    },
    {
      "name": "nvidia_fp8_capability",
      "status": "not_run",
      "detail": "A real FP8 matrix multiplication completed on NVIDIA CUDA."
    },
    {
      "name": "ascend_fp8_capability",
      "status": "not_run",
      "detail": "A real FP8 matrix multiplicati

In [5]:
import gc, json, sys
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/baseline_quality.json'
BASELINE_REPORT = PROJECT_DIR / 'out/baseline_quality_experiment/baseline_quality_report.json'
INT8_MANIFEST = RESULT_DIR / 'smoothquant-int8-w8a8/quantized_model_manifest.json'
sys.path.insert(0, str(PROJECT_DIR))

from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.evaluation.finqa_metrics import evaluate_numeric_answers
from src.quantization.int8_fp8 import (
    check_structured_json, evaluate_int8_outputs,
    load_smoothquant_service, validate_int8_evaluation,
)
if not INT8_MANIFEST.is_file():
    raise RuntimeError('没有真实 INT8 manifest；请先运行 int8_fp8_experiment_colab.ipynb。')
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
baseline_metrics = json.loads(BASELINE_REPORT.read_text(encoding='utf-8'))['metrics']
int8_service = load_smoothquant_service(BASELINE_CONFIG, INT8_MANIFEST, MODEL_MANIFEST, RESULT_DIR / 'int8_quality_service.jsonl')
fp16_service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'fp16_comparison_service.jsonl')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Inserted 606 quantizers


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
predictions, references = [], []
batch_size = int(quality_config['batch_size'])
for start in range(0, len(assets.quality_dev), batch_size):
    rows = [assets.quality_dev[index] for index in range(start, min(start + batch_size, len(assets.quality_dev)))]
    requests = [InferenceRequest.from_finqa_row(row, request_id=f'int8-quality-{start + offset + 1:06d}') for offset, row in enumerate(rows)]
    results = int8_service.generate_batch(requests, prompt_suffix=quality_config.get('prompt_suffix', ''))
    predictions.extend(result['output_text'] for result in results)
    references.extend(str(row.get('answer') or row.get('qa', {}).get('answer')) for row in rows)
print('完整 dev 样本完成:', len(predictions))

Loading extension modelopt_cuda_ext...
Loaded extension modelopt_cuda_ext in 58.6 seconds
完整 dev 样本完成: 883


In [7]:
rows = [dict(row) for row in assets.quality_dev]
rows.sort(key=lambda row: len(InferenceRequest.from_finqa_row(row).render_prompt()), reverse=True)
long_results = []
for index, row in enumerate(rows[:3]):
    reference = str(row.get('answer') or row.get('qa', {}).get('answer'))
    request = InferenceRequest.from_finqa_row(row, request_id=f'long-{index + 1}')
    fp16_text = fp16_service.generate_one(request)['output_text']
    int8_text = int8_service.generate_one(request)['output_text']
    fp16_ok = evaluate_numeric_answers([fp16_text], [reference])['correct'] == 1
    int8_ok = evaluate_numeric_answers([int8_text], [reference])['correct'] == 1
    long_results.append({'index': index, 'input_characters': len(request.render_prompt()), 'baseline_passed': fp16_ok, 'passed': int8_ok})

structured_results = []
suffix = '\nReturn JSON with fields answer and unit only.'
for index, row in enumerate(rows[:10]):
    request = InferenceRequest.from_finqa_row(row, request_id=f'structured-{index + 1}')
    fp16_check = check_structured_json(fp16_service.generate_one(request, prompt_suffix=suffix)['output_text'], ['answer', 'unit'])
    int8_check = check_structured_json(int8_service.generate_one(request, prompt_suffix=suffix)['output_text'], ['answer', 'unit'])
    structured_results.append({'index': index, 'baseline_passed': fp16_check['passed'], **int8_check})

In [8]:
payload = evaluate_int8_outputs(
    predictions, references, long_results, structured_results,
    tolerance=float(quality_config.get('tolerance', 1e-4)),
    baseline_numeric=baseline_metrics,
)
evaluation_path = RESULT_DIR / 'evaluation.json'
evaluation_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = validate_int8_evaluation(evaluation_path)
validation = report.to_dict()
(RESULT_DIR / 'evaluation_validation.json').write_text(json.dumps(validation, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(payload, ensure_ascii=False, indent=2))
print(json.dumps(validation, ensure_ascii=False, indent=2))
assert report.complete, 'INT8 三类质量对比未完整执行。'
print('TODO 6.4 数字、长上下文、结构化输出对比：PASS')
gc.collect()

{
  "schema_version": 2,
  "status": "completed",
  "numeric": {
    "total": 883,
    "parsed": 848,
    "correct": 6,
    "parse_rate": 0.9603624009060022,
    "numeric_accuracy": 0.006795016987542469
  },
  "numeric_comparison": {
    "baseline_numeric_accuracy": 0.009060022650056626,
    "int8_numeric_accuracy": 0.006795016987542469,
    "absolute_change": -0.002265005662514157
  },
  "long_context": {
    "total": 3,
    "passed": 0,
    "pass_rate": 0.0,
    "cases": [
      {
        "index": 0,
        "input_characters": 9219,
        "baseline_passed": false,
        "passed": false
      },
      {
        "index": 1,
        "input_characters": 9205,
        "baseline_passed": false,
        "passed": false
      },
      {
        "index": 2,
        "input_characters": 9198,
        "baseline_passed": false,
        "passed": false
      }
    ]
  },
  "structured_output": {
    "total": 10,
    "passed": 0,
    "pass_rate": 0.0,
    "cases": [
      {
        "index": 0,

108

## 第 7 章：int8_fp8_evaluation_colab.ipynb

原始文件：`notebooks/int8_fp8_evaluation_colab.ipynb`


# TODO 6.4：INT8 质量对比

对真实 SmoothQuant INT8 状态执行完整 FinQA dev 数值评测，并使用同一批样本对比 FP16/INT8 的长上下文数值正确性和 JSON 结构化输出成功率。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/baseline_quality.json'
BASELINE_REPORT = PROJECT_DIR / 'out/baseline_quality_experiment/baseline_quality_report.json'
INT8_MANIFEST = RESULT_DIR / 'smoothquant-int8-w8a8/quantized_model_manifest.json'
sys.path.insert(0, str(PROJECT_DIR))

from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.evaluation.finqa_metrics import evaluate_numeric_answers
from src.quantization.int8_fp8 import (
    check_structured_json, evaluate_int8_outputs,
    load_smoothquant_service, validate_int8_evaluation,
)
if not INT8_MANIFEST.is_file():
    raise RuntimeError('没有真实 INT8 manifest；请先运行 int8_fp8_experiment_colab.ipynb。')
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
baseline_metrics = json.loads(BASELINE_REPORT.read_text(encoding='utf-8'))['metrics']
int8_service = load_smoothquant_service(BASELINE_CONFIG, INT8_MANIFEST, MODEL_MANIFEST, RESULT_DIR / 'int8_quality_service.jsonl')
fp16_service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'fp16_comparison_service.jsonl')

In [ ]:
predictions, references = [], []
batch_size = int(quality_config['batch_size'])
for start in range(0, len(assets.quality_dev), batch_size):
    rows = [assets.quality_dev[index] for index in range(start, min(start + batch_size, len(assets.quality_dev)))]
    requests = [InferenceRequest.from_finqa_row(row, request_id=f'int8-quality-{start + offset + 1:06d}') for offset, row in enumerate(rows)]
    results = int8_service.generate_batch(requests, prompt_suffix=quality_config.get('prompt_suffix', ''))
    predictions.extend(result['output_text'] for result in results)
    references.extend(str(row.get('answer') or row.get('qa', {}).get('answer')) for row in rows)
print('完整 dev 样本完成:', len(predictions))

In [ ]:
rows = [dict(row) for row in assets.quality_dev]
rows.sort(key=lambda row: len(InferenceRequest.from_finqa_row(row).render_prompt()), reverse=True)
long_results = []
for index, row in enumerate(rows[:3]):
    reference = str(row.get('answer') or row.get('qa', {}).get('answer'))
    request = InferenceRequest.from_finqa_row(row, request_id=f'long-{index + 1}')
    fp16_text = fp16_service.generate_one(request)['output_text']
    int8_text = int8_service.generate_one(request)['output_text']
    fp16_ok = evaluate_numeric_answers([fp16_text], [reference])['correct'] == 1
    int8_ok = evaluate_numeric_answers([int8_text], [reference])['correct'] == 1
    long_results.append({'index': index, 'input_characters': len(request.render_prompt()), 'baseline_passed': fp16_ok, 'passed': int8_ok})

structured_results = []
suffix = '\nReturn JSON with fields answer and unit only.'
for index, row in enumerate(rows[:10]):
    request = InferenceRequest.from_finqa_row(row, request_id=f'structured-{index + 1}')
    fp16_check = check_structured_json(fp16_service.generate_one(request, prompt_suffix=suffix)['output_text'], ['answer', 'unit'])
    int8_check = check_structured_json(int8_service.generate_one(request, prompt_suffix=suffix)['output_text'], ['answer', 'unit'])
    structured_results.append({'index': index, 'baseline_passed': fp16_check['passed'], **int8_check})

In [ ]:
payload = evaluate_int8_outputs(
    predictions, references, long_results, structured_results,
    tolerance=float(quality_config.get('tolerance', 1e-4)),
    baseline_numeric=baseline_metrics,
)
evaluation_path = RESULT_DIR / 'evaluation.json'
evaluation_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = validate_int8_evaluation(evaluation_path)
validation = report.to_dict()
(RESULT_DIR / 'evaluation_validation.json').write_text(json.dumps(validation, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(payload, ensure_ascii=False, indent=2))
print(json.dumps(validation, ensure_ascii=False, indent=2))
assert report.complete, 'INT8 三类质量对比未完整执行。'
print('TODO 6.4 数字、长上下文、结构化输出对比：PASS')
gc.collect()

## 第 8 章：quantization_comparison_benchmark_colab.ipynb

原始文件：`notebooks/quantization_comparison_benchmark_colab.ipynb`


# TODO 6.5：在 A100 40GB 补跑 FP16 / INT8 Benchmark

已有 AWQ/GPTQ 结果来自 `NVIDIA A100-SXM4-40GB`。本 Notebook 只补跑 FP16 和 SmoothQuant INT8，并复用已有六份 INT4 Benchmark，避免重新量化或重新运行 AWQ/GPTQ。

运行前请在 Colab 选择 A100 GPU。初始化单元会严格检查设备名称必须为 `NVIDIA A100-SXM4-40GB`；如果分配到 80GB 版本会立即停止。全部单元执行完成后，将更新的 `out/quantization_comparison` 和 `out/int8_fp8/benchmark_*.json` 拉回项目。

In [2]:
%pip -q install -U "nvidia-modelopt[hf]" accelerate safetensors
print('安装完成后请重启 Colab 会话，再从下一单元继续。')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 28.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 135.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.4 MB/s eta 0:00:00
安装完成后请重启 Colab 会话，再从下一单元继续。


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
INT8_MANIFEST = PROJECT_DIR / 'out/int8_fp8/smoothquant-int8-w8a8/quantized_model_manifest.json'
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
BENCHMARK_CONFIG = PROJECT_DIR / 'configs/quantization_benchmark.json'
COMPARISON_CONFIG = PROJECT_DIR / 'configs/quantization_comparison.json'
RESULT_DIR = PROJECT_DIR / 'out/quantization_comparison'
INT8_RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
EXPECTED_GPU = 'NVIDIA A100-SXM4-40GB'
sys.path.insert(0, str(PROJECT_DIR))

from src.baseline.service import BaselineService, InferenceRequest
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.data.finqa_assets import load_assets
from src.quantization.comparison import build_quantization_comparison, render_markdown, validate_quantization_comparison
from src.quantization.int8_fp8 import load_smoothquant_service

assert torch.cuda.is_available(), '请切换到 NVIDIA A100 GPU。'
actual_gpu = torch.cuda.get_device_name(0)
assert actual_gpu == EXPECTED_GPU, f'需要 {EXPECTED_GPU}，当前为 {actual_gpu}。请重新分配运行时。'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
INT8_RESULT_DIR.mkdir(parents=True, exist_ok=True)
assets = load_assets(DATASET_DIR)
benchmark_config = json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
prompt = InferenceRequest.from_finqa_row(assets.quality_dev[0], request_id='comparison-benchmark').render_prompt()
expected_case_count = (
    len(benchmark_config['matrix']['batch_sizes'])
    * len(benchmark_config['matrix']['input_lengths'])
    * len(benchmark_config['matrix']['output_lengths'])
)
assert expected_case_count == 16, f'预期 16 个测试点，实际为 {expected_case_count}'
print('GPU:', actual_gpu)
print('测试点:', expected_case_count)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
测试点: 16


In [2]:
int4_reports = sorted((PROJECT_DIR / 'out/awq').glob('*/comparison_benchmark_report.json'))
int4_reports += sorted((PROJECT_DIR / 'out/gptq').glob('*/comparison_benchmark_report.json'))
assert len(int4_reports) == 6, f'应找到 6 份 AWQ/GPTQ 报告，实际找到 {len(int4_reports)} 份'
for report_path in int4_reports:
    existing_report = json.loads(report_path.read_text(encoding='utf-8'))
    report_gpu = existing_report['environment']['gpu']
    report_case_count = len(existing_report['cases'])
    assert report_gpu == EXPECTED_GPU, f'{report_path}: GPU 为 {report_gpu}'
    assert report_case_count == expected_case_count, f'{report_path}: 测试点为 {report_case_count}'
    print('可复用:', report_path.relative_to(PROJECT_DIR), report_gpu, report_case_count)

可复用: out/awq/awq-int4-w4a16-g128/comparison_benchmark_report.json NVIDIA A100-SXM4-40GB 16
可复用: out/awq/awq-int4-w4a16-g32/comparison_benchmark_report.json NVIDIA A100-SXM4-40GB 16
可复用: out/awq/awq-int4-w4a16-g64/comparison_benchmark_report.json NVIDIA A100-SXM4-40GB 16
可复用: out/gptq/gptq-int4-w4a16-g128/comparison_benchmark_report.json NVIDIA A100-SXM4-40GB 16
可复用: out/gptq/gptq-int4-w4a16-g32/comparison_benchmark_report.json NVIDIA A100-SXM4-40GB 16
可复用: out/gptq/gptq-int4-w4a16-g64/comparison_benchmark_report.json NVIDIA A100-SXM4-40GB 16


In [3]:
fp16_service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'fp16_a100_service.jsonl')
fp16_report = run_benchmark(fp16_service, prompt, benchmark_config)
fp16_path = RESULT_DIR / 'fp16_a100_benchmark_report.json'
fp16_path.write_text(json.dumps(fp16_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
fp16_validation = validate_benchmark_report(fp16_path)
(RESULT_DIR / 'fp16_a100_benchmark_validation.json').write_text(json.dumps(fp16_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert fp16_validation.complete
assert fp16_report['environment']['gpu'] == EXPECTED_GPU
del fp16_service
gc.collect()
torch.cuda.empty_cache()
print('FP16 A100 40GB Benchmark: PASS')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
FP16 A100 40GB Benchmark: PASS


In [4]:
int8_service = load_smoothquant_service(BASELINE_CONFIG, INT8_MANIFEST, MODEL_MANIFEST, INT8_RESULT_DIR / 'int8_benchmark_service.jsonl')
int8_report = run_benchmark(int8_service, prompt, benchmark_config)
int8_path = INT8_RESULT_DIR / 'benchmark_report.json'
int8_path.write_text(json.dumps(int8_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
int8_validation = validate_benchmark_report(int8_path)
(INT8_RESULT_DIR / 'benchmark_validation.json').write_text(json.dumps(int8_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert int8_validation.complete
assert int8_report['environment']['gpu'] == EXPECTED_GPU
del int8_service
gc.collect()
torch.cuda.empty_cache()
print('SmoothQuant INT8 A100 40GB Benchmark: PASS')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Registered <class 'transformers.models.qwen2.modeling_qwen2.Qwen2Attention'> to _QuantAttention for KV Cache quantization
Inserted 606 quantizers
Benchmark 1/16: bs1-in128-out16
Loading extension modelopt_cuda_ext...
Loaded extension modelopt_cuda_ext in 56.0 seconds
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
SmoothQuant INT8 A100 40GB Benchmark: PASS


In [5]:
report = build_quantization_comparison(PROJECT_DIR, COMPARISON_CONFIG)
report_path = RESULT_DIR / 'comparison_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(RESULT_DIR / 'comparison_report.md').write_text(render_markdown(report), encoding='utf-8')
validation = validate_quantization_comparison(report_path)
validation_dict = validation.to_dict()
(RESULT_DIR / 'validation.json').write_text(json.dumps(validation_dict, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
checks = {item['name']: item['status'] for item in validation_dict['checks']}
for check_name in ('memory_changes_recorded', 'latency_changes_recorded', 'throughput_changes_recorded'):
    assert checks[check_name] == 'pass', f'{check_name}: {checks[check_name]}'
print(json.dumps(validation_dict, ensure_ascii=False, indent=2))
print('性能三项已完成；若单位 token 成本仍为 not_run，只需补充 GPU 小时单价。')
print('请拉回:', RESULT_DIR)
print('请拉回:', int8_path)
print('请拉回:', INT8_RESULT_DIR / 'benchmark_validation.json')

{
  "report": "/content/drive/MyDrive/OptimizationSystem/out/quantization_comparison/comparison_report.json",
  "complete": false,
  "checks": [
    {
      "name": "fp16_int8_int4_fp8_comparison_table",
      "status": "pass",
      "detail": "FP16, INT8, INT4, and FP8 rows are present."
    },
    {
      "name": "quality_changes_recorded",
      "status": "pass",
      "detail": "Every candidate has measured quality."
    },
    {
      "name": "memory_changes_recorded",
      "status": "pass",
      "detail": "Every candidate has measured memory."
    },
    {
      "name": "latency_changes_recorded",
      "status": "pass",
      "detail": "Every candidate has measured latency."
    },
    {
      "name": "throughput_changes_recorded",
      "status": "pass",
      "detail": "Every candidate has measured throughput."
    },
    {
      "name": "unit_token_cost_recorded",
      "status": "not_run",
      "detail": "Every measured candidate has an approved monetary cost."
    },
   

## 第 9 章：quality_repair_6_7_2_colab.ipynb

原始文件：`notebooks/quality_repair_6_7_2_colab.ipynb`


# TODO 6.7.2：Prompt 与确定性计算消融

在同一个 FP16 模型和 883 条 FinQA dev 上比较四种 Prompt 方案。few-shot 示例只从 calibration/train 读取；不要把 dev/test 答案放入 Prompt。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/quality_repair.json'
RESULT_DIR = PROJECT_DIR / 'out/quality_repair_6_7_2'
sys.path.insert(0, str(PROJECT_DIR))
from src.baseline.service import BaselineService
from src.data.finqa_assets import load_assets
from src.evaluation.quality_repair import run_quality_repair_experiment

assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
dataset_manifest = json.loads((DATASET_DIR / 'manifest.json').read_text(encoding='utf-8'))
service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'quality_service.jsonl')
report = run_quality_repair_experiment(
    service,
    assets.quality_dev,
    dataset_manifest,
    quality_config,
    RESULT_DIR,
    calibration=assets.calibration,
)
print(json.dumps(report, ensure_ascii=False, indent=2))
del service
gc.collect()
torch.cuda.empty_cache()

Mounted at /content/drive


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[6.7.2] 方案 1/3 zero_shot_final_answer 开始，总进度 0.0%
[6.7.2] zero_shot_final_answer: 4/80 (5.0%), 总进度 1.7%
[6.7.2] zero_shot_final_answer: 8/80 (10.0%), 总进度 3.3%
[6.7.2] zero_shot_final_answer: 12/80 (15.0%), 总进度 5.0%
[6.7.2] zero_shot_final_answer: 16/80 (20.0%), 总进度 6.7%
[6.7.2] zero_shot_final_answer: 20/80 (25.0%), 总进度 8.3%
[6.7.2] zero_shot_final_answer: 24/80 (30.0%), 总进度 10.0%
[6.7.2] zero_shot_final_answer: 28/80 (35.0%), 总进度 11.7%
[6.7.2] zero_shot_final_answer: 32/80 (40.0%), 总进度 13.3%
[6.7.2] zero_shot_final_answer: 36/80 (45.0%), 总进度 15.0%
[6.7.2] zero_shot_final_answer: 40/80 (50.0%), 总进度 16.7%
[6.7.2] zero_shot_final_answer: 44/80 (55.0%), 总进度 18.3%
[6.7.2] zero_shot_final_answer: 48/80 (60.0%), 总进度 20.0%
[6.7.2] zero_shot_final_answer: 52/80 (65.0%), 总进度 21.7%
[6.7.2] zero_shot_final_answer: 56/80 (70.0%), 总进度 23.3%
[6.7.2] zero_shot_final_answer: 60/80 (75.0%), 总进度 25.0%
[6.7.2] zero_shot_final_answer: 64/80 (80.0%), 总进度 26.7%
[6.7.2] zero_shot_final_answer: 68/80 (85.0%),

In [3]:
import gc
import json
import subprocess
import sys
from pathlib import Path

import torch
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub'])
from huggingface_hub import snapshot_download

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_real')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/quality_repair.json'
RESULT_ROOT = PROJECT_DIR / 'out/quality_repair_6_7_3_model_scale'
sys.path.insert(0, str(PROJECT_DIR))

MODEL_DOWNLOADS = {
    'qwen2.5-1.5b': ('Qwen/Qwen2.5-1.5B-Instruct', 'qwen2.5-1.5b-instruct'),
    'qwen2.5-3b': ('Qwen/Qwen2.5-3B-Instruct', 'qwen2.5-3b-instruct'),
}
def model_download_complete(model_dir):
    has_config = (model_dir / 'config.json').exists()
    has_weights = (model_dir / 'model.safetensors').exists() or (model_dir / 'model.safetensors.index.json').exists()
    return has_config and has_weights

for model_key, (repo_id, directory_name) in MODEL_DOWNLOADS.items():
    target_dir = MODEL_ROOT / directory_name
    if model_download_complete(target_dir):
        print(f'{model_key}: 已存在，跳过下载 -> {target_dir}')
    else:
        print(f'{model_key}: 开始下载 {repo_id} -> {target_dir}')
        target_dir.mkdir(parents=True, exist_ok=True)
        snapshot_download(repo_id=repo_id, local_dir=str(target_dir))
        print(f'{model_key}: 下载完成')

from src.baseline.service import BaselineService
from src.data.finqa_assets import load_assets
from src.evaluation.quality_repair import run_quality_repair_experiment

assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'

MODEL_CANDIDATES = {
    'qwen2.5-1.5b': [
        MODEL_ROOT / 'qwen2.5-1.5b-instruct',
        MODEL_ROOT / 'Qwen2.5-1.5B-Instruct',
    ],
    'qwen2.5-3b': [
        MODEL_ROOT / 'qwen2.5-3b-instruct',
        MODEL_ROOT / 'Qwen2.5-3B-Instruct',
    ],
}
MODEL_IDS = {
    'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'qwen2.5-3b': 'Qwen/Qwen2.5-3B-Instruct',
}

def resolve_model_dir(candidates):
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    expected = '\n'.join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f'找不到模型目录，请检查以下路径：\n{expected}')

def write_local_manifest(model_key, model_dir, output_dir):
    model_id = MODEL_IDS[model_key]
    manifest = {
        'schema_version': 1,
        'model_id': model_id,
        'model_path': str(model_dir),
        'version': model_id,
        'commit': 'colab-local',
        'weight_files': [],
        'config_file': 'config.json',
        'tokenizer_files': [],
        'expected_sha256': {},
        'runtime': {
            'framework': 'transformers',
            'model_class': 'causal_lm',
            'dtype': 'float16',
            'device': 'cuda:0',
            'trust_remote_code': False,
        },
    }
    manifest_path = output_dir / 'model_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return manifest_path

assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
dataset_manifest = json.loads((DATASET_DIR / 'manifest.json').read_text(encoding='utf-8'))
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

all_reports = {}
for model_key, candidates in MODEL_CANDIDATES.items():
    model_dir = resolve_model_dir(candidates)
    model_result_dir = RESULT_ROOT / model_key
    model_result_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = write_local_manifest(model_key, model_dir, model_result_dir)
    print(f'\n===== {MODEL_IDS[model_key]} =====')
    print(f'模型目录: {model_dir}')
    print(f'结果目录: {model_result_dir}')
    service = BaselineService.from_local_model(
        BASELINE_CONFIG,
        manifest_path,
        model_result_dir / 'quality_service.jsonl',
    )
    report = run_quality_repair_experiment(
        service,
        assets.quality_dev,
        dataset_manifest,
        quality_config,
        model_result_dir,
        calibration=assets.calibration,
    )
    report['model'] = {
        'key': model_key,
        'model_id': MODEL_IDS[model_key],
        'model_path': str(model_dir),
    }
    (model_result_dir / 'comparison_report.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    all_reports[model_key] = report
    for variant in report['variants']:
        metrics = variant['metrics']
        print(
            f"{variant['variant']}: parse={metrics['parse_rate']:.2%}, "
            f"accuracy={metrics['numeric_accuracy']:.2%}, "
            f"correct={metrics['correct']}/{metrics['total']}"
        )
    del service
    gc.collect()
    torch.cuda.empty_cache()

print('\n===== 模型规模对比完成 =====')
for model_key, report in all_reports.items():
    print(model_key, '->', RESULT_ROOT / model_key / 'comparison_report.json')

qwen2.5-1.5b: 开始下载 Qwen/Qwen2.5-1.5B-Instruct -> /content/drive/MyDrive/banking_llm_project/model_input_real/qwen2.5-1.5b-instruct


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

qwen2.5-1.5b: 下载完成
qwen2.5-3b: 开始下载 Qwen/Qwen2.5-3B-Instruct -> /content/drive/MyDrive/banking_llm_project/model_input_real/qwen2.5-3b-instruct


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

qwen2.5-3b: 下载完成

===== Qwen/Qwen2.5-1.5B-Instruct =====
模型目录: /content/drive/MyDrive/banking_llm_project/model_input_real/qwen2.5-1.5b-instruct
结果目录: /content/drive/MyDrive/OptimizationSystem/out/quality_repair_6_7_3_model_scale/qwen2.5-1.5b


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[6.7.2] 方案 1/3 zero_shot_final_answer 开始，总进度 0.0%
[6.7.2] zero_shot_final_answer: 4/80 (5.0%), 总进度 1.7%
[6.7.2] zero_shot_final_answer: 8/80 (10.0%), 总进度 3.3%
[6.7.2] zero_shot_final_answer: 12/80 (15.0%), 总进度 5.0%
[6.7.2] zero_shot_final_answer: 16/80 (20.0%), 总进度 6.7%
[6.7.2] zero_shot_final_answer: 20/80 (25.0%), 总进度 8.3%
[6.7.2] zero_shot_final_answer: 24/80 (30.0%), 总进度 10.0%
[6.7.2] zero_shot_final_answer: 28/80 (35.0%), 总进度 11.7%
[6.7.2] zero_shot_final_answer: 32/80 (40.0%), 总进度 13.3%
[6.7.2] zero_shot_final_answer: 36/80 (45.0%), 总进度 15.0%
[6.7.2] zero_shot_final_answer: 40/80 (50.0%), 总进度 16.7%
[6.7.2] zero_shot_final_answer: 44/80 (55.0%), 总进度 18.3%
[6.7.2] zero_shot_final_answer: 48/80 (60.0%), 总进度 20.0%
[6.7.2] zero_shot_final_answer: 52/80 (65.0%), 总进度 21.7%
[6.7.2] zero_shot_final_answer: 56/80 (70.0%), 总进度 23.3%
[6.7.2] zero_shot_final_answer: 60/80 (75.0%), 总进度 25.0%
[6.7.2] zero_shot_final_answer: 64/80 (80.0%), 总进度 26.7%
[6.7.2] zero_shot_final_answer: 68/80 (85.0%),

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[6.7.2] 方案 1/3 zero_shot_final_answer 开始，总进度 0.0%
[6.7.2] zero_shot_final_answer: 4/80 (5.0%), 总进度 1.7%
[6.7.2] zero_shot_final_answer: 8/80 (10.0%), 总进度 3.3%
[6.7.2] zero_shot_final_answer: 12/80 (15.0%), 总进度 5.0%
[6.7.2] zero_shot_final_answer: 16/80 (20.0%), 总进度 6.7%
[6.7.2] zero_shot_final_answer: 20/80 (25.0%), 总进度 8.3%
[6.7.2] zero_shot_final_answer: 24/80 (30.0%), 总进度 10.0%
[6.7.2] zero_shot_final_answer: 28/80 (35.0%), 总进度 11.7%
[6.7.2] zero_shot_final_answer: 32/80 (40.0%), 总进度 13.3%
[6.7.2] zero_shot_final_answer: 36/80 (45.0%), 总进度 15.0%
[6.7.2] zero_shot_final_answer: 40/80 (50.0%), 总进度 16.7%
[6.7.2] zero_shot_final_answer: 44/80 (55.0%), 总进度 18.3%
[6.7.2] zero_shot_final_answer: 48/80 (60.0%), 总进度 20.0%
[6.7.2] zero_shot_final_answer: 52/80 (65.0%), 总进度 21.7%
[6.7.2] zero_shot_final_answer: 56/80 (70.0%), 总进度 23.3%
[6.7.2] zero_shot_final_answer: 60/80 (75.0%), 总进度 25.0%
[6.7.2] zero_shot_final_answer: 64/80 (80.0%), 总进度 26.7%
[6.7.2] zero_shot_final_answer: 68/80 (85.0%),

## 第 10 章：quality_repair_6_7_3_model_scale_colab.ipynb

原始文件：`notebooks/quality_repair_6_7_3_model_scale_colab.ipynb`


# TODO 6.7.3：1.5B 与 3B 模型小规模质量对比

本 Notebook 保留 0.5B 的 80 条 pilot 结果作为控制组，在相同 80 条样本、相同 Prompt 和相同生成参数下依次运行 1.5B 与 3B。

代码单元会从 Hugging Face 下载公开权重并保存到 Google Drive；如果目录名不同，只需修改 `MODEL_CANDIDATES`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc
import json
import subprocess
import sys
from pathlib import Path

import torch
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub'])
from huggingface_hub import snapshot_download

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_real')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/quality_repair.json'
RESULT_ROOT = PROJECT_DIR / 'out/quality_repair_6_7_3_model_scale'
sys.path.insert(0, str(PROJECT_DIR))

MODEL_DOWNLOADS = {
    'qwen2.5-1.5b': ('Qwen/Qwen2.5-1.5B-Instruct', 'qwen2.5-1.5b-instruct'),
    'qwen2.5-3b': ('Qwen/Qwen2.5-3B-Instruct', 'qwen2.5-3b-instruct'),
}
def model_download_complete(model_dir):
    has_config = (model_dir / 'config.json').exists()
    has_weights = (model_dir / 'model.safetensors').exists() or (model_dir / 'model.safetensors.index.json').exists()
    return has_config and has_weights

for model_key, (repo_id, directory_name) in MODEL_DOWNLOADS.items():
    target_dir = MODEL_ROOT / directory_name
    if model_download_complete(target_dir):
        print(f'{model_key}: 已存在，跳过下载 -> {target_dir}')
    else:
        print(f'{model_key}: 开始下载 {repo_id} -> {target_dir}')
        target_dir.mkdir(parents=True, exist_ok=True)
        snapshot_download(repo_id=repo_id, local_dir=str(target_dir))
        print(f'{model_key}: 下载完成')

from src.baseline.service import BaselineService
from src.data.finqa_assets import load_assets
from src.evaluation.quality_repair import run_quality_repair_experiment

assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'

MODEL_CANDIDATES = {
    'qwen2.5-1.5b': [
        MODEL_ROOT / 'qwen2.5-1.5b-instruct',
        MODEL_ROOT / 'Qwen2.5-1.5B-Instruct',
    ],
    'qwen2.5-3b': [
        MODEL_ROOT / 'qwen2.5-3b-instruct',
        MODEL_ROOT / 'Qwen2.5-3B-Instruct',
    ],
}
MODEL_IDS = {
    'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'qwen2.5-3b': 'Qwen/Qwen2.5-3B-Instruct',
}

def resolve_model_dir(candidates):
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    expected = '\n'.join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f'找不到模型目录，请检查以下路径：\n{expected}')

def write_local_manifest(model_key, model_dir, output_dir):
    model_id = MODEL_IDS[model_key]
    manifest = {
        'schema_version': 1,
        'model_id': model_id,
        'model_path': str(model_dir),
        'version': model_id,
        'commit': 'colab-local',
        'weight_files': [],
        'config_file': 'config.json',
        'tokenizer_files': [],
        'expected_sha256': {},
        'runtime': {
            'framework': 'transformers',
            'model_class': 'causal_lm',
            'dtype': 'float16',
            'device': 'cuda:0',
            'trust_remote_code': False,
        },
    }
    manifest_path = output_dir / 'model_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return manifest_path

assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
dataset_manifest = json.loads((DATASET_DIR / 'manifest.json').read_text(encoding='utf-8'))
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

all_reports = {}
for model_key, candidates in MODEL_CANDIDATES.items():
    model_dir = resolve_model_dir(candidates)
    model_result_dir = RESULT_ROOT / model_key
    model_result_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = write_local_manifest(model_key, model_dir, model_result_dir)
    print(f'\n===== {MODEL_IDS[model_key]} =====')
    print(f'模型目录: {model_dir}')
    print(f'结果目录: {model_result_dir}')
    service = BaselineService.from_local_model(
        BASELINE_CONFIG,
        manifest_path,
        model_result_dir / 'quality_service.jsonl',
    )
    report = run_quality_repair_experiment(
        service,
        assets.quality_dev,
        dataset_manifest,
        quality_config,
        model_result_dir,
        calibration=assets.calibration,
    )
    report['model'] = {
        'key': model_key,
        'model_id': MODEL_IDS[model_key],
        'model_path': str(model_dir),
    }
    (model_result_dir / 'comparison_report.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    all_reports[model_key] = report
    for variant in report['variants']:
        metrics = variant['metrics']
        print(
            f"{variant['variant']}: parse={metrics['parse_rate']:.2%}, "
            f"accuracy={metrics['numeric_accuracy']:.2%}, "
            f"correct={metrics['correct']}/{metrics['total']}"
        )
    del service
    gc.collect()
    torch.cuda.empty_cache()

print('\n===== 模型规模对比完成 =====')
for model_key, report in all_reports.items():
    print(model_key, '->', RESULT_ROOT / model_key / 'comparison_report.json')